<a href="https://colab.research.google.com/github/RRADJon/TEMPO/blob/main/PBPK_Boltz2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Make sure to visit https://build.nvidia.com/mit/boltz2 and select "get API key"
include this API key in your secrets tab under NVIDIA_API_KEY and turn on notebook access.

In [1]:
#@title BLOCK 1 — Setup NVIDIA NIM Boltz-2
!pip install -q httpx fastapi nest_asyncio py3Dmol

import asyncio
import json
import logging
import sys
import os
from typing import Any, Dict, Optional
from pathlib import Path
import httpx
from fastapi import HTTPException
from google.colab import userdata
import nest_asyncio
import requests
import base64

nest_asyncio.apply()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Retrieve API key
try:
    NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
except Exception:
    print("Error: Add 'NVIDIA_API_KEY' in Secrets tab.")
    sys.exit(1)

STATUS_URL = "https://api.nvcf.nvidia.com/v2/nvcf/pexec/status/{task_id}"
PUBLIC_URL = "https://health.api.nvidia.com/v1/biology/mit/boltz2/predict"

headers = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Content-Type": "application/json"
}

In [2]:
ligand = "CC(=O)NC1=CC=C(C=C1)O"#@param {type:"string"}


In [3]:
#@title Template Mapping

template_pdb_map = {
    # Serum Carriers
    "Albumin": "1Ao6",
    "AGP": "3KQ0",

    # ABC transporters
    "ABCB1": "AF_AFP08183F1",
    "ABCB11": "6LR0",
    "ABCC2": "8jx7",
    "ABCC3": "8hw2",
    "ABCC4": "8xok",
    "ABCC5": "8wi0",
    "ABCC6": "AF_AFO95255F1",
    "ABCG2": "6vxi",

    # CYP enzymes
    "CYP1A2": "2HI4",
    "CYP2A6": "3t3q",
    "CYP2B6": "5uda",
    "CYP2C18": "2h6p",
    "CYP2C19": "4gqs",
    "CYP2C8": "1pq2",
    "CYP2C9": "3qi8",
    "CYP2D6": "4wnt",
    "CYP2E1": "3e4e",
    "CYP3A5": "6mjm",
    "CYP4A11": "AF_AFQ02928F1",

    # New Additions
    "TRPV4": "4dx1",
    "CYP3A4": "1TQN",
    "COX1": "6Y3C",
    "COX2": "5F19",
    "SIK3": "8OKU",
    "HRH1": "3RZE",
    "KCNA1": "AF_AFQ09470F1",
    "KCNA2": "AF_AFP16389F1",
    "TAU": "6ODG",
    "ADRB1": "7BU6",
    "ADRB2": "1GQ4",
    "HMGCR": "1DQ8",

    # Transporters
    "NTCP": "7pqq",
    "OAT4": "8wjh",
    "SLC22A1": "AF_AFO15245F1",
    "SLC22A2": "AF_AFO15244F1",
    "SLC22A4": "AF_AFQ9H015F1",
    "SLC22A5": "9pfb",
    "SLC22A6": "9m9v",
    "SLC22A7": "AF_AFQ9Y694F1",
    "SLC22A8": "AF_AFQ8TCC7F1",
    "SLC22A9": "AF_AFQ8IVM8F1",
    "SLC47A1": "9r1e",
    "SLC47A2": "AF_AFQ86VL8F1",
    "SLCO1A2": "AF_AFP46721F1",
    "SLCO1B1": "8phw",
    "SLCO1B3": "8pg0",
    "SLCO2B1": "AF_AFO94956F1",
    "SLCO4C1": "AF_AFQ6ZQN7F1",

    # UGT
    "UGT1A1": "AF_AFP22309F1",
    "UGT1A4": "AF_AFP22310F1",
    "UGT1A6": "AF_AFP19224F1",
    "UGT1A9": "AF_AFO60656F1",
    "UGT2B4": "AF_AFP06133F1",
    "UGT2B7": "AF_AFP16662F1"
}

In [4]:
#@title Short Template Mapping

template_pdb_map = {
    "Albumin": "1Ao6",
    "AGP": "3KQ0",
    "ABCB1": "AF_AFP08183F1"
}

In [5]:
#@title Download and Encode Templates

def download_pdb(pdb_id):
    filename = f"{pdb_id}.pdb"
    if not os.path.exists(filename):
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        r = requests.get(url)
        with open(filename, "wb") as f:
            f.write(r.content)
    return filename

def load_pdb_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

template_b64_map = {}

for protein_id, pdb_id in template_pdb_map.items():
    pdb_file = download_pdb(pdb_id)
    template_b64_map[protein_id] = load_pdb_base64(pdb_file)

print("Templates ready.")

Templates ready.


In [6]:
#@title Protein Sequences Batch
protein_sequences = [
    {
        "id": "Albumin",
        "sequence": "MKWVTFISLLFLFSSAYSRGVFRRDAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL"
    },
    {
        "id": "AGP",
        "sequence": "MALSWVLTVLSLLPLLEAQIPLCANLVPVPITNATLDRITGKWFYIASAFRNEEYNKSVQEIQATFFYFTPNKTEDTIFLREYQTRQDQCIYNTTYLNVQRENGTISRYVGGQEHFAHLLILRDTKTYMLAFDVNDEKNWGLSVYADKPETTKEQLGEFYEALDCLRIPKSDVVYTDWKKDKCEPLEKQHEKERKQEEGES"
    },
    {
        "id": "ABCB1",
        "sequence": "MDLEGDRNGGAKKKNFFKLNNKSEKDKKEKKPTVSVFSMFRYSNWLDKLYMVVGTLAAIIHGAGLPLMMLVFGEMTDIFANAGNLEDLMSNITNRSDINDTGFFMNLEEDMTRYAYYYSGIGAGVLVAAYIQVSFWCLAAGRQIHKIRKQFFHAIMRQEIGWFDVHDVGELNTRLTDDVSKINEGIGDKIGMFFQSMATFFTGFIVGFTRGWKLTLVILAISPVLGLSAAVWAKILSSFTDKELLAYAKAGAVAEEVLAAIRTVIAFGGQKKELERYNKNLEEAKRIGIKKAITANISIGAAFLLIYASYALAFWYGTTLVLSGEYSIGQVLTVFFSVLIGAFSVGQASPSIEAFANARGAAYEIFKIIDNKPSIDSYSKSGHKPDNIKGNLEFRNVHFSYPSRKEVKILKGLNLKVQSGQTVALVGNSGCGKSTTVQLMQRLYDPTEGMVSVDGQDIRTINVRFLREIIGVVSQEPVLFATTIAENIRYGRENVTMDEIEKAVKEANAYDFIMKLPHKFDTLVGERGAQLSGGQKQRIAIARALVRNPKILLLDEATSALDTESEAVVQVALDKARKGRTTIVIAHRLSTVRNADVIAGFDDGVIVEKGNHDELMKEKGIYFKLVTMQTAGNEVELENAADESKSEIDALEMSSNDSRSSLIRKRSTRRSVRGSQAQDRKLSTKEALDESIPPVSFWRIMKLNLTEWPYFVVGVFCAIINGGLQPAFAIIFSKIIGVFTRIDDPETKRQNSNLFSLLFLALGIISFITFFLQGFTFGKAGEILTKRLRYMVFRSMLRQDVSWFDDPKNTTGALTTRLANDAAQVKGAIGSRLAVITQNIANLGTGIIISFIYGWQLTLLLLAIVPIIAIAGVVEMKMLSGQALKDKKELEGAGKIATEAIENFRTVVSLTQEQKFEHMYAQSLQVPYRNSLRKAHIFGITFSFTQAMMYFSYAGCFRFGAYLVAHKLMSFEDVLLVFSAVVFGAMAVGQVSSFAPDYAKAKISAAHIIMIIEKTPLIDSYSTEGLMPNTLEGNVTFGEVVFNYPTRPDIPVLQGLSLEVKKGQTLALVGSSGCGKSTVVQLLERFYDPLAGKVLLDGKEIKRLNVQWLRAHLGIVSQEPILFDCSIAENIAYGDNSRVVSQEEIVRAAKEANIHAFIESLPNKYSTKVGDKGTQLSGGQKQRIAIARALVRQPHILLLDEATSALDTESEKVVQEALDKAREGRTCIVIAHRLSTIQNADLIVVFQNGRVKEHGTHQQLLAQKGIYFSMVSVQAGTKRQ"
    },
    {
        "id": "ABCB11",
        "sequence": "MSDSVILRSIKKFGEENDGFESDKSYNNDKKSRLQDEKKGDGVRVGFFQLFRFSSSTDIWLMFVGSLCAFLHGIAQPGVLLIFGTMTDVFIDYDVELQELQIPGKACVNNTIVWTNSSLNQNMTNGTRCGLLNIESEMIKFASYYAGIAVAVLITGYIQICFWVIAAARQIQKMRKFYFRRIMRMEIGWFDCNSVGELNTRFSDDINKINDAIADQMALFIQRMTSTICGFLLGFFRGWKLTLVIISVSPLIGIGAATIGLSVSKFTDYELKAYAKAGVVADEVISSMRTVAAFGGEKREVERYEKNLVFAQRWGIRKGIVMGFFTGFVWCLIFLCYALAFWYGSTLVLDEGEYTPGTLVQIFLSVIVGALNLGNASPCLEAFATGRAAATSIFETIDRKPIIDCMSEDGYKLDRIKGEIEFHNVTFHYPSRPEVKILNDLNMVIKPGEMTALVGPSGAGKSTALQLIQRFYDPCEGMVTVDGHDIRSLNIQWLRDQIGIVEQEPVLFSTTIAENIRYGREDATMEDIVQAAKEANAYNFIMDLPQQFDTLVGEGGGQMSGGQKQRVAIARALIRNPKILLLDMATSALDNESEAMVQEVLSKIQHGHTIISVAHRLSTVRAADTIIGFEHGTAVERGTHEELLERKGVYFTLVTLQSQGNQALNEEDIKDATEDDMLARTFSRGSYQDSLRASIRQRSKSQLSYLVHEPPLAVVDHKSTYEEDRKDKDIPVQEEVEPAPVRRILKFSAPEWPYMLVGSVGAAVNGTVTPLYAFLFSQILGTFSIPDKEEQRSQINGVCLLFVAMGCVSLFTQFLQGYAFAKSGELLTKRLRKFGFRAMLGQDIAWFDDLRNSPGALTTRLATDASQVQGAAGSQIGMIVNSFTNVTVAMIIAFSFSWKLSLVILCFFPFLALSGATQTRMLTGFASRDKQALEMVGQITNEALSNIRTVAGIGKERRFIEALETELEKPFKTAIQKANIYGFCFAFAQCIMFIANSASYRYGGYLISNEGLHFSYVFRVISAVVLSATALGRAFSYTPSYAKAKISAARFFQLLDRQPPISVYNTAGEKWDNFQGKIDFVDCKFTYPSRPDSQVLNGLSVSISPGQTLAFVGSSGCGKSTSIQLLERFYDPDQGKVMIDGHDSKKVNVQFLRSNIGIVSQEPVLFACSIMDNIKYGDNTKEIPMERVIAAAKQAQLHDFVMSLPEKYETNVGSQGSQLSRGEKQRIAIARAIVRDPKILLLDEATSALDTESEKTVQVALDKAREGRTCIVIAHRLSTIQNADIIAVMAQGVVIEKGTHEELMAQKGAYYKLVTTGSPIS"
    },
    {
        "id": "ABCC2",
        "sequence": "MLEKFCNSTFWNSSFLDSPEADLPLCFEQTVLVWIPLGYLWLLAPWQLLHVYKSRTKRSSTTKLYLAKQVFVGFLLILAAIELALVLTEDSGQATVPAVRYTNPSLYLGTWLLVLLIQYSRQWCVQKNSWFLSLFWILSILCGTFQFQTLIRTLLQGDNSNLAYSCLFFISYGFQILILIFSAFSENNESSNNPSSIASFLSSITYSWYDSIILKGYKRPLTLEDVWEVDEEMKTKTLVSKFETHMKRELQKARRALQRRQEKSSQQNSGARLPGLNKNQSQSQDALVLEDVEKKKKKSGTKKDVPKSWLMKALFKTFYMVLLKSFLLKLVNDIFTFVSPQLLKLLISFASDRDTYLWIGYLCAILLFTAALIQSFCLQCYFQLCFKLGVKVRTAIMASVYKKALTLSNLARKEYTVGETVNLMSVDAQKLMDVTNFMHMLWSSVLQIVLSIFFLWRELGPSVLAGVGVMVLVIPINAILSTKSKTIQVKNMKNKDKRLKIMNEILSGIKILKYFAWEPSFRDQVQNLRKKELKNLLAFSQLQCVVIFVFQLTPVLVSVVTFSVYVLVDSNNILDAQKAFTSITLFNILRFPLSMLPMMISSMLQASVSTERLEKYLGGDDLDTSAIRHDCNFDKAMQFSEASFTWEHDSEATVRDVNLDIMAGQLVAVIGPVGSGKSSLISAMLGEMENVHGHITIKGTTAYVPQQSWIQNGTIKDNILFGTEFNEKRYQQVLEACALLPDLEMLPGGDLAEIGEKGINLSGGQKQRISLARATYQNLDIYLLDDPLSAVDAHVGKHIFNKVLGPNGLLKGKTRLLVTHSMHFLPQVDEIVVLGNGTIVEKGSYSALLAKKGEFAKNLKTFLRHTGPEEEATVHDGSEEEDDDYGLISSVEEIPEDAASITMRRENSFRRTLSRSSRSNGRHLKSLRNSLKTRNVNSLKEDEELVKGQKLIKKEFIETGKVKFSIYLEYLQAIGLFSIFFIILAFVMNSVAFIGSNLWLSAWTSDSKIFNSTDYPASQRDMRVGVYGALGLAQGIFVFIAHFWSAFGFVHASNILHKQLLNNILRAPMRFFDTTPTGRIVNRFAGDISTVDDTLPQSLRSWITCFLGIISTLVMICMATPVFTIIVIPLGIIYVSVQMFYVSTSRQLRRLDSVTRSPIYSHFSETVSGLPVIRAFEHQQRFLKHNEVRIDTNQKCVFSWITSNRWLAIRLELVGNLTVFFSALMMVIYRDTLSGDTVGFVLSNALNITQTLNWLVRMTSEIETNIVAVERITEYTKVENEAPWVTDKRPPPDWPSKGKIQFNNYQVRYRPELDLVLRGITCDIGSMEKIGVVGRTGAGKSSLTNCLFRILEAAGGQIIIDGVDIASIGLHDLREKLTIIPQDPILFSGSLRMNLDPFNNYSDEEIWKALELAHLKSFVASLQLGLSHEVTEAGGNLSIGQRQLLCLGRALLRKSKILVLDEATAAVDLETDNLIQTTIQNEFAHCTVITIAHRLHTIMDSDKVMVLDNGKIIECGSPEELLQIPGPFYFMAKEAGIENVNSTKF"
    },
    {
        "id": "ABCC3",
        "sequence": "MDALCGSGELGSKFWDSNLSVHTENPDLTPCFQNSLLAWVPCIYLWVALPCYLLYLRHHCRGYIILSHLSKLKMVLGVLLWCVSWADLFYSFHGLVHGRAPAPVFFVTPLVVGVTMLLATLLIQYERLQGVQSSGVLIIFWFLCVVCAIVPFRSKILLAKAEGEISDPFRFTTFYIHFALVLSALILACFREKPPFFSAKNVDPNPYPETSAGFLSRLFFWWFTKMAIYGYRHPLEEKDLWSLKEEDRSQMVVQQLLEAWRKQEKQTARHKASAAPGKNASGEDEVLLGARPRPRKPSFLKALLATFGSSFLISACFKLIQDLLSFINPQLLSILIRFISNPMAPSWWGFLVAGLMFLCSMMQSLILQHYYHYIFVTGVKFRTGIMGVIYRKALVITNSVKRASTVGEIVNLMSVDAQRFMDLAPFLNLLWSAPLQIILAIYFLWQNLGPSVLAGVAFMVLLIPLNGAVAVKMRAFQVKQMKLKDSRIKLMSEILNGIKVLKLYAWEPSFLKQVEGIRQGELQLLRTAAYLHTTTTFTWMCSPFLVTLITLWVYVYVDPNNVLDAEKAFVSVSLFNILRLPLNMLPQLISNLTQASVSLKRIQQFLSQEELDPQSVERKTISPGYAITIHSGTFTWAQDLPPTLHSLDIQVPKGALVAVVGPVGCGKSSLVSALLGEMEKLEGKVHMKGSVAYVPQQAWIQNCTLQENVLFGKALNPKRYQQTLEACALLADLEMLPGGDQTEIGEKGINLSGGQRQRVSLARAVYSDADIFLLDDPLSAVDSHVAKHIFDHVIGPEGVLAGKTRVLVTHGISFLPQTDFIIVLADGQVSEMGPYPALLQRNGSFANFLCNYAPDEDQGHLEDSWTALEGAEDKEALLIEDTLSNHTDLTDNDPVTYVVQKQFMRQLSALSSDGEGQGRPVPRRHLGPSEKVQVTEAKADGALTQEEKAAIGTVELSVFWDYAKAVGLCTTLAICLLYVGQSAAAIGANVWLSAWTNDAMADSRQNNTSLRLGVYAALGILQGFLVMLAAMAMAAGGIQAARVLHQALLHNKIRSPQSFFDTTPSGRILNCFSKDIYVVDEVLAPVILMLLNSFFNAISTLVVIMASTPLFTVVILPLAVLYTLVQRFYAATSRQLKRLESVSRSPIYSHFSETVTGASVIRAYNRSRDFEIISDTKVDANQRSCYPYIISNRWLSIGVEFVGNCVVLFAALFAVIGRSSLNPGLVGLSVSYSLQVTFALNWMIRMMSDLESNIVAVERVKEYSKTETEAPWVVEGSRPPEGWPPRGEVEFRNYSVRYRPGLDLVLRDLSLHVHGGEKVGIVGRTGAGKSSMTLCLFRILEAAKGEIRIDGLNVADIGLHDLRSQLTIIPQDPILFSGTLRMNLDPFGSYSEEDIWWALELSHLHTFVSSQPAGLDFQCSEGGENLSVGQRQLVCLARALLRKSRILVLDEATAAIDLETDNLIQATIRTQFDTCTVLTIAHRLNTIMDYTRVLVLDKGVVAEFDSPANLIAARGIFYGMARDAGLA"
    },
    {
        "id": "ABCC4",
        "sequence": "MLPVYQEVKPNPLQDANLCSRVFFWWLNPLFKIGHKRRLEEDDMYSVLPEDRSQHLGEELQGFWDKEVLRAENDAQKPSLTRAIIKCYWKSYLVLGIFTLIEESAKVIQPIFLGKIINYFENYDPMDSVALNTAYAYATVLTFCTLILAILHHLYFYHVQCAGMRLRVAMCHMIYRKALRLSNMAMGKTTTGQIVNLLSNDVNKFDQVTVFLHFLWAGPLQAIAVTALLWMEIGISCLAGMAVLIILLPLQSCFGKLFSSLRSKTATFTDARIRTMNEVITGIRIIKMYAWEKSFSNLITNLRKKEISKILRSSCLRGMNLASFFSASKIIVFVTFTTYVLLGSVITASRVFVAVTLYGAVRLTVTLFFPSAIERVSEAIVSIRRIQTFLLLDEISQRNRQLPSDGKKMVHVQDFTAFWDKASETPTLQGLSFTVRPGELLAVVGPVGAGKSSLLSAVLGELAPSHGLVSVHGRIAYVSQQPWVFSGTLRSNILFGKKYEKERYEKVIKACALKKDLQLLEDGDLTVIGDRGTTLSGGQKARVNLARAVYQDADIYLLDDPLSAVDAEVSRHLFELCICQILHEKITILVTHQLQYLKAASQILILKDGKMVQKGTYTEFLKSGIDFGSLLKKDNEESEQPPVPGTPTLRNRTFSESSVWSQQSSRPSLKDGALESQDTENVPVTLSEENRSEGKVGFQAYKNYFRAGAHWIVFIFLILLNTAAQVAYVLQDWWLSYWANKQSMLNVTVNGGGNVTEKLDLNWYLGIYSGLTVATVLFGIARSLLVFYVLVNSSQTLHNKMFESILKAPVLFFDRNPIGRILNRFSKDIGHLDDLLPLTFLDFIQTLLQVVGVVSVAVAVIPWIAIPLVPLGIIFIFLRRYFLETSRDVKRLESTTRSPVFSHLSSSLQGLWTIRAYKAEERCQELFDAHQDLHSEAWFLFLTTSRWFAVRLDAICAMFVIIVAFGSLILAKTLDAGQVGLALSYALTLMGMFQWCVRQSAEVENMMISVERVIEYTDLEKEAPWEYQKRPPPAWPHEGVIIFDNVNFMYSPGGPLVLKHLTALIKSQEKVGIVGRTGAGKSSLISALFRLSEPEGKIWIDKILTTEIGLHDLRKKMSIIPQEPVLFTGTMRKNLDPFNEHTDEELWNALQEVQLKETIEDLPGKMDTELAESGSNFSVGQRQLVCLARAILRKNQILIIDEATANVDPRTDELIQKKIREKFAHCTVLTIAHRLNTIIDSDKIMVLDSGRLKEYDEPYVLLQNKESLFYKMVQQLGKAEAAALTETAKQVYFKRNYPHIGHTDHMVTNTSNGQPSTLTIFETAL"
    },
    {
        "id": "ABCC5",
        "sequence": "MKDIDIGKEYIIPSPGYRSVRERTSTSGTHRDREDSKFRRTRPLECQDALETAARAEGLSLDASMHSQLRILDEEHPKGKYHHGLSALKPIRTTSKHQHPVDNAGLFSCMTFSWLSSLARVAHKKGELSMEDVWSLSKHESSDVNCRRLERLWQEELNEVGPDAASLRRVVWIFCRTRLILSIVCLMITQLAGFSGPAFMVKHLLEYTQATESNLQYSLLLVLGLLLTEIVRSWSLALTWALNYRTGVRLRGAILTMAFKKILKLKNIKEKSLGELINICSNDGQRMFEAAAVGSLLAGGPVVAILGMIYNVIILGPTGFLGSAVFILFYPAMMFASRLTAYFRRKCVAATDERVQKMNEVLTYIKFIKMYAWVKAFSQSVQKIREEERRILEKAGYFQSITVGVAPIVVVIASVVTFSVHMTLGFDLTAAQAFTVVTVFNSMTFALKVTPFSVKSLSEASVAVDRFKSLFLMEEVHMIKNKPASPHIKIEMKNATLAWDSSHSSIQNSPKLTPKMKKDKRASRGKKEKVRQLQRTEHQAVLAEQKGHLLLDSDERPSPEEEEGKHIHLGHLRLQRTLHSIDLEIQEGKLVGICGSVGSGKTSLISAILGQMTLLEGSIAISGTFAYVAQQAWILNATLRDNILFGKEYDEERYNSVLNSCCLRPDLAILPSSDLTEIGERGANLSGGQRQRISLARALYSDRSIYILDDPLSALDAHVGNHIFNSAIRKHLKSKTVLFVTHQLQYLVDCDEVIFMKEGCITERGTHEELMNLNGDYATIFNNLLLGETPPVEINSKKETSGSQKKSQDKGPKTGSVKKEKAVKPEEGQLVQLEEKGQGSVPWSVYGVYIQAAGGPLAFLVIMALFMLNVGSTAFSTWWLSYWIKQGSGNTTVTRGNETSVSDSMKDNPHMQYYASIYALSMAVMLILKAIRGVVFVKGTLRASSRLHDELFRRILRSPMKFFDTTPTGRILNRFSKDMDEVDVRLPFQAEMFIQNVILVFFCVGMIAGVFPWFLVAVGPLVILFSVLHIVSRVLIRELKRLDNITQSPFLSHITSSIQGLATIHAYNKGQEFLHRYQELLDDNQAPFFLFTCAMRWLAVRLDLISIALITTTGLMIVLMHGQIPPAYAGLAISYAVQLTGLFQFTVRLASETEARFTSVERINHYIKTLSLEAPARIKNKAPSPDWPQEGEVTFENAEMRYRENLPLVLKKVSFTIKPKEKIGIVGRTGSGKSSLGMALFRLVELSGGCIKIDGVRISDIGLADLRSKLSIIPQEPVLFSGTVRSNLDPFNQYTEDQIWDALERTHMKECIAQLPLKLESEVMENGDNFSVGERQLLCIARALLRHCKILILDEATAAMDTETDLLIQETIREAFADCTMLTIAHRLHTVLGSDRIMVLAQGQVVEFDTPSVLLSNDSSRFYAMFAAAENKVAVKG"
    },
    {
        "id": "ABCC6",
        "sequence": "MAAPAEPCAGQGVWNQTEPEPAATSLLSLCFLRTAGVWVPPMYLWVLGPIYLLFIHHHGRGYLRMSPLFKAKMVLGFALIVLCTSSVAVALWKIQQGTPEAPEFLIHPTVWLTTMSFAVFLIHTERKKGVQSSGVLFGYWLLCFVLPATNAAQQASGAGFQSDPVRHLSTYLCLSLVVAQFVLSCLADQPPFFPEDPQQSNPCPETGAAFPSKATFWWVSGLVWRGYRRPLRPKDLWSLGRENSSEELVSRLEKEWMRNRSAARRHNKAIAFKRKGGSGMKAPETEPFLRQEGSQWRPLLKAIWQVFHSTFLLGTLSLIISDVFRFTVPKLLSLFLEFIGDPKPPAWKGYLLAVLMFLSACLQTLFEQQNMYRLKVLQMRLRSAITGLVYRKVLALSSGSRKASAVGDVVNLVSVDVQRLTESVLYLNGLWLPLVWIVVCFVYLWQLLGPSALTAIAVFLSLLPLNFFISKKRNHHQEEQMRQKDSRARLTSSILRNSKTIKFHGWEGAFLDRVLGIRGQELGALRTSGLLFSVSLVSFQVSTFLVALVVFAVHTLVAENAMNAEKAFVTLTVLNILNKAQAFLPFSIHSLVQARVSFDRLVTFLCLEEVDPGVVDSSSSGSAAGKDCITIHSATFAWSQESPPCLHRINLTVPQGCLLAVVGPVGAGKSSLLSALLGELSKVEGFVSIEGAVAYVPQEAWVQNTSVVENVCFGQELDPPWLERVLEACALQPDVDSFPEGIHTSIGEQGMNLSGGQKQRLSLARAVYRKAAVYLLDDPLAALDAHVGQHVFNQVIGPGGLLQGTTRILVTHALHILPQADWIIVLANGAIAEMGSYQELLQRKGALMCLLDQARQPGDRGEGETEPGTSTKDPRGTSAGRRPELRRERSIKSVPEKDRTTSEAQTEVPLDDPDRAGWPAGKDSIQYGRVKATVHLAYLRAVGTPLCLYALFLFLCQQVASFCRGYWLSLWADDPAVGGQQTQAALRGGIFGLLGCLQAIGLFASMAAVLLGGARASRLLFQRLLWDVVRSPISFFERTPIGHLLNRFSKETDTVDVDIPDKLRSLLMYAFGLLEVSLVVAVATPLATVAILPLFLLYAGFQSLYVVSSCQLRRLESASYSSVCSHMAETFQGSTVVRAFRTQAPFVAQNNARVDESQRISFPRLVADRWLAANVELLGNGLVFAAATCAVLSKAHLSAGLVGFSVSAALQVTQTLQWVVRNWTDLENSIVSVERMQDYAWTPKEAPWRLPTCAAQPPWPQGGQIEFRDFGLRYRPELPLAVQGVSFKIHAGEKVGIVGRTGAGKSSLASGLLRLQEAAEGGIWIDGVPIAHVGLHTLRSRISIIPQDPILFPGSLRMNLDLLQEHSDEAIWAALETVQLKALVASLPGQLQYKCADRGEDLSVGQKQLLCLARALLRKTQILILDEATAAVDPGTELQMQAMLGSWFAQCTVLLIAHRLRSVMDCARVLVMDKGQVAESGSPAQLLAQKGLFYRLAQESGLV"
    },
    {
        "id": "COX1", #PTGS1
        "sequence": "MSRSLLLWFLLFLLLLPPLPVLLADPGAPTPVNPCCYYPCQHQGICVRFGLDRYQCDCTRTGYSGPNCTIPGLWTWLRNSLRPSPSFTHFLLTHGRWFWEFVNATFIREMLMRLVLTVRSNLIPSPPTYNSAHDYISWESFSNVSYYTRILPSVPKDCPTPMGTKGKKQLPDAQLLARRFLLRRKFIPDPQGTNLMFAFFAQHFTHQFFKTSGKMGPGFTKALGHGVDLGHIYGDNLERQYQLRLFKDGKLKYQVLDGEMYPPSVEEAPVLMHYPRGIPPQSQMAVGQEVFGLLPGLMLYATLWLREHNRVCDLLKAEHPTWGDEQLFQTTRLILIGETIKIVIEEYVQQLSGYFLQLKFDPELLFGVQFQYRNRIAMEFNHLYHWHPLMPDSFKVGSQEYSYEQFLFNTSMLVDYGVEALVDAFSRQIAGRIGGGRNMDHHILHVAVDVIRESREMRLQPFNEYRKRFGMKPYTSFQELVGEKEMAAELEELYGDIDALEFYPGLLLEKCHPNSIFGESMIEIGAPFSLKGLLGNPICSPEYWKPSTFGGEVGFNIVKTATLKKLVCLNTKTCPYVSFRVPDASQDDGPAVERPSTEL"
    },
    {
        "id": "COX2", #PTGS2
        "sequence": "MLARALLLCAVLALSHTANPCCSHPCQNRGVCMSVGFDQYKCDCTRTGFYGENCSTPEFLTRIKLFLKPTPNTVHYILTHFKGFWNVVNNIPFLRNAIMSYVLTSRSHLIDSPPTYNADYGYKSWEAFSNLSYYTRALPPVPDDCPTPLGVKGKKQLPDSNEIVEKLLLRRKFIPDPQGSNMMFAFFAQHFTHQFFKTDHKRGPAFTNGLGHGVDLNHIYGETLARQRKLRLFKDGKMKYQIIDGEMYPPTVKDTQAEMIYPPQVPEHLRFAVGQEVFGLVPGLMMYATIWLREHNRVCDVLKQEHPEWGDEQLFQTSRLILIGETIKIVIEDYVQHLSGYHFKLKFDPELLFNKQFQYQNRIAAEFNTLYHWHPLLPDTFQIHDQKYNYQQFIYNNSILLEHGITQFVESFTRQIAGRVAGGRNVPPAVQKVSQASIDQSRQMKYQSFNEYRKRFMLKPYESFEELTGEKEMSAELEALYGDIDAVELYPALLVEKPRPDAIFGETMVEVGAPFSLKGLMGNVICSPAYWKPSTFGGEVGFQIINTASIQSLICNNVKGCPFTSFSVPDPELIKTVTINASSSRSGLDDINPTVLLKERSTEL"
    },
    {
        "id": "SIK3",
        "sequence": "MAAAAASGAGGAAGAGTGGAGPAGRLLPPPAPGSPAAPAAVSPAAGQPRPPAPASRGPMPARIGYYEIDRTIGKGNFAVVKRATHLVTKAKVAIKIIDKTQLDEENLKKIFREVQIMKMLCHPHIIRLYQVMETERMIYLVTEYASGGEIFDHLVAHGRMAEKEARRKFKQIVTAVYFCHCRNIVHRDLKAENLLLDANLNIKIADFGFSNLFTPGQLLKTWCGSPPYAAPELFEGKEYDGPKVDIWSLGVVLYVLVCGALPFDGSTLQNLRARVLSGKFRIPFFMSTECEHLIRHMLVLDPNKRLSMEQICKHKWMKLGDADPNFDRLIAECQQLKEERQVDPLNEDVLLAMEDMGLDKEQTLQSLRSDAYDHYSAIYSLLCDRHKRHKTLRLGALPSMPRALAFQAPVNIQAEQAGTAMNISVPQVQLINPENQIVEPDGTLNLDSDEGEEPSPEALVRYLSMRRHTVGVADPRTEVMEDLQKLLPGFPGVNPQAPFLQVAPNVNFMHNLLPMQNLQPTGQLEYKEQSLLQPPTLQLLNGMGPLGRRASDGGANIQLHAQQLLKRPRGPSPLVTMTPAVPAVTPVDEESSDGEPDQEAVQSSTYKDSNTLHLPTERFSPVRRFSDGAASIQAFKAHLEKMGNNSSIKQLQQECEQLQKMYGGQIDERTLEKTQQQHMLYQQEQHHQILQQQIQDSICPPQPSPPLQAACENQPALLTHQLQRLRIQPSSPPPNHPNNHLFRQPSNSPPPMSSAMIQPHGAASSSQFQGLPSRSAIFQQQPENCSSPPNVALTCLGMQQPAQSQQVTIQVQEPVDMLSNMPGTAAGSSGRGISISPSAGQMQMQHRTNLMATLSYGHRPLSKQLSADSAEAHSLNVNRFSPANYDQAHLHPHLFSDQSRGSPSSYSPSTGVGFSPTQALKVPPLDQFPTFPPSAHQQPPHYTTSALQQALLSPTPPDYTRHQQVPHILQGLLSPRHSLTGHSDIRLPPTEFAQLIKRQQQQRQQQQQQQQQQEYQELFRHMNQGDAGSLAPSLGGQSMTERQALSYQNADSYHHHTSPQHLLQIRAQECVSQASSPTPPHGYAHQPALMHSESMEEDCSCEGAKDGFQDSKSSSTLTKGCHDSPLLLSTGGPGDPESLLGTVSHAQELGIHPYGHQPTAAFSKNKVPSREPVIGNCMDRSSPGQAVELPDHNGLGYPARPSVHEHHRPRALQRHHTIQNSDDAYVQLDNLPGMSLVAGKALSSARMSDAVLSQSSLMGSQQFQDGENEECGASLGGHEHPDLSDGSQHLNSSCYPSTCITDILLSYKHPEVSFSMEQAGV"
    },
    {
        "id": "HRH1",
        "sequence": "MSLPNSSCLLEDKMCEGNKTTMASPQLMPLVVVLSTICLVTVGLNLLVLYAVRSERKLHTVGNLYIVSLSVADLIVGAVVMPMNILYLLMSKWSLGRPLCLFWLSMDYVASTASIFSVFILCIDRYRSVQQPLRYLKYRTKTRASATILGAWFLSFLWVIPILGWNHFMQQTSVRREDKCETDFYDVTWFKVMTAIINFYLPTLLMLWFYAKIYKAVRQHCQHRELINRSLPSFSEIKLRPENPKGDAKKPGKESPWEVLKRKPKDAGGGSVLKSPSQTPKEMKSPVVFSQEDDREVDKLYCFPLDIVHMQAAAEGSSRDYVAVNRSHGQLKTDEQGLNTHGASEISEDQMLGDSQSFSRTDSDTTTETAPGKGKLRSGSNTGLDYIKFTWKRLRSHSRQYVSGLHMNRERKAAKQLGFIMAAFILCWIPYFIFFMVIAFCKNCCNEHLHMFTIWLGYINSTLNPLIYPLCNENFKKTFKRILHIRS"
    },
    {
        "id": "KCNA1",
        "sequence": "MTVMSGENVDEASAAPGHPQDGSYPRQADHDDHECCERVVINISGLRFETQLKTLAQFPNTLLGNPKKRMRYFDPLRNEYFFDRNRPSFDAILYYYQSGGRLRRPVNVPLDMFSEEIKFYELGEEAMEKFREDEGFIKEEERPLPEKEYQRQVWLLFEYPESSGPARVIAIVSVMVILISIVIFCLETLPELKDDKDFTGTVHRIDNTTVIYNSNIFTDPFFIVETLCIIWFSFELVVRFFACPSKTDFFKNIMNFIDIVAIIPYFITLGTEIAEQEGNQKGEQATSLAILRVIRLVRVFRIFKLSRHSKGLQILGQTLKASMRELGLLIFFLFIGVILFSSAVYFAEAEEAESHFSSIPDAFWWAVVSMTTVGYGDMYPVTIGGKIVGSLCAIAGVLTIALPVPVIVSNFNYFYHRETEGEEQAQLLHVSSPNLASDSDLSRRSSSTMSKSEYMEIEEDMNNSIAHYRQVNIRTANCTTANQNCVNKSKLLTDV"
    },
    {
        "id": "KCNA2",
        "sequence": "MTVATGDPADEAAALPGHPQDTYDPEADHECCERVVINISGLRFETQLKTLAQFPETLLGDPKKRMRYFDPLRNEYFFDRNRPSFDAILYYYQSGGRLRRPVNVPLDIFSEEIRFYELGEEAMEMFREDEGYIKEEERPLPENEFQRQVWLLFEYPESSGPARIIAIVSVMVILISIVSFCLETLPIFRDENEDMHGSGVTFHTYSNSTIGYQQSTSFTDPFFIVETLCIIWFSFEFLVRFFACPSKAGFFTNIMNIIDIVAIIPYFITLGTELAEKPEDAQQGQQAMSLAILRVIRLVRVFRIFKLSRHSKGLQILGQTLKASMRELGLLIFFLFIGVILFSSAVYFAEADERESQFPSIPDAFWWAVVSMTTVGYGDMVPTTIGGKIVGSLCAIAGVLTIALPVPVIVSNFNYFYHRETEGEEQAQYLQVTSCPKIPSSPDLKKSRSASTISKSDYMEIQEGVNNSNEDFREENLKTANCTLANTNYVNITKMLTDV"
    },
    {
        "id": "TAU",
        "sequence": "MAEPRQEFEVMEDHAGTYGLGDRKDQGGYTMHQDQEGDTDAGLKESPLQTPTEDGSEEPGSETSDAKSTPTAEDVTAPLVDEGAPGKQAAAQPHTEIPEGTTAEEAGIGDTPSLEDEAAGHVTQEPESGKVVQEGFLREPGPPGLSHQLMSGMPGAPLLPEGPREATRQPSGTGPEDTEGGRHAPELLKHQLLGDLHQEGPPLKGAGGKERPGSKEEVDEDRDVDESSPQDSPPSKASPAQDGRPPQTAAREATSIPGFPAEGAIPLPVDFLSKVSTEIPASEPDGPSVGRAKGQDAPLEFTFHVEITPNVQKEQAHSEEHLGRAAFPGAPGEGPEARGPSLGEDTKEADLPEPSEKQPAAAPRGKPVSRVPQLKARMVSKSKDGTGSDDKKAKTSTRSSAKTLKNRPCLSPKHPTPGSSDPLIQPSSPAVCPEPPSSPKYVSSVTSRTGSSGAKEMKLKGADGKTKIATPRGAAPPGQKGQANATRIPAKTPPAPKTPPSSGEPPKSGDRSGYSSPGSPGTPGSRSRTPSLPTPPTREPKKVAVVRTPPKSPSSAKSRLQTAPVPMPDLKNVKSKIGSTENLKHQPGGGKVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYKPVDLSKVTSKCGSLGNIHHKPGGGQVEVKSEKLDFKDRVQSKIGSLDNITHVPGGGNKKIETHKLTFRENAKAKTDHGAEIVYKSPVVSGDTSPRHLSNVSSTGSIDMVDSPQLATLADEVSASLAKQGL"
    },
    {
        "id": "ADRB1",
        "sequence": "MGAGVLVLGASEPGNLSSAAPLPDGAATAARLLVPASPPASLLPPASESPEPLSQQWTAGMGLLMALIVLLIVAGNVLVIVAIAKTPRLQTLTNLFIMSLASADLVMGLLVVPFGATIVVWGRWEYGSFFCELWTSVDVLCVTASIETLCVIALDRYLAITSPFRYQSLLTRARARGLVCTVWAISALVSFLPILMHWWRAESDEARRCYNDPKCCDFVTNRAYAIASSVVSFYVPLCIMAFVYLRVFREAQKQVKKIDSCERRFLGGPARPPSPSPSPVPAPAPPPGPPRPAAAAATAPLANGRAGKRRPSRLVALREQKALKTLGIIMGVFTLCWLPFFLANVVKAFHRELVPDRLFVFFNWLGYANSAFNPIIYCRSPDFRKAFQGLLCCARRAARRRHATHGDRPRASGCLARPGPPPSPGAASDDDDDDVVGATPPARLLEPWAGCNGGAAADSDSSLDEPCRPGFASESKV"
    },
    {
        "id": "ADRB2",
        "sequence": "MGQPGNGSAFLLAPNGSHAPDHDVTQERDEVWVVGMGIVMSLIVLAIVFGNVLVITAIAKFERLQTVTNYFITSLACADLVMGLAVVPFGAAHILMKMWTFGNFWCEFWTSIDVLCVTASIETLCVIAVDRYFAITSPFKYQSLLTKNKARVIILMVWIVSGLTSFLPIQMHWYRATHQEAINCYANETCCDFFTNQAYAIASSIVSFYVPLVIMVFVYSRVFQEAKRQLQKIDKSEGRFHVQNLSQVEQDGRTGHGLRRSSKFCLKEHKALKTLGIIMGTFTLCWLPFFIVNIVHVIQDNLIRKEVYILLNWIGYVNSGFNPLIYCRSPDFRIAFQELLCLRRSSLKAYGNGYSSNGNTGEQSGYHVEQEKENKLLCEDLPGTEDFVGHQGTVPSDNIDSQGRNCSTNDSLL"
    },
    {
        "id": "HMGCR",
        "sequence": "MLSRLFRMHGLFVASHPWEVIVGTVTLTICMMSMNMFTGNNKICGWNYECPKFEEDVLSSDIIILTITRCIAILYIYFQFQNLRQLGSKYILGIAGLFTIFSSFVFSTVVIHFLDKELTGLNEALPFFLLLIDLSRASTLAKFALSSNSQDEVRENIARGMAILGPTFTLDALVECLVIGVGTMSGVRQLEIMCCFGCMSVLANYFVFMTFFPACVSLVLELSRESREGRPIWQLSHFARVLEEEENKPNPVTQRVKMIMSLGLVLVHAHSRWIADPSPQNSTADTSKVSLGLDENVSKRIEPSVSLWQFYLSKMISMDIEQVITLSLALLLAVKYIFFEQTETESTLSLKNPITSPVVTQKKVPDNCCRREPMLVRNNQKCDSVEEETGINRERKVEVIKPLVAETDTPNRATFVVGNSSLLDTSSVLVTQEPEIELPREPRPNEECLQILGNAEKGAKFLSDAEIIQLVNAKHIPAYKLETLMETHERGVSIRRQLLSKKLSEPSSLQYLPYRDYNYSLVMGACCENVIGYMPIPVGVAGPLCLDEKEFQVPMATTEGCLVASTNRGCRAIGLGGGASSRVLADGMTRGPVVRLPRACDSAEVKAWLETSEGFAVIKEAFDSTSRFARLQKLHTSIAGRNLYIRFQSRSGDAMGMNMISKGTEKALSKLHEYFPEMQILAVSGNYCTDKKPAAINWIEGRGKSVVCEAVIPAKVVREVLKTTTEAMIEVNINKNLVGSAMAGSIGGYNAHAANIVTAIYIACGQDAAQNVGSSNCITLMEASGPTNEDLYISCTMPSIEIGTVGGGTNLLPQQACLQMLGVQGACKDNPGENARQLARIVCGTVMAGELSLMAALAAGHLVKSHMIHNRSKINLQDLQGACTKKTA"
    },
    {
        "id": "ABCG2",
        "sequence": "MSSSNVEVFIPVSQGNTNGFPATASNDLKAFTEGAVLSFHNICYRVKLKSGFLPCRKPVEKEILSNINGIMKPGLNAILGPTGGGKSSLLDVLAARKDPSGLSGDVLINGAPRPANFKCNSGYVVQDDVVMGTLTVRENLQFSAALRLATTMTNHEKNERINRVIQELGLDKVADSKVGTQFIRGVSGGERKRTSIGMELITDPSILFLDEPTTGLDSSTANAVLLLLKRMSKQGRTIIFSIHQPRYSIFKLFDSLTLLASGRLMFHGPAQEALGYFESAGYHCEAYNNPADFFLDIINGDSTAVALNREEDFKATEIIEPSKQDKPLIEKLAEIYVNSSFYKETKAELHQLSGGEKKKKITVFKEISYTTSFCHQLRWVSKRSFKNLLGNPQASIAQIIVTVVLGLVIGAIYFGLKNDSTGIQNRAGVLFFLTTNQCFSSVSAVELFVVEKKLFIHEYISGYYRVSSYFLGKLLSDLLPMRMLPSIIFTCIVYFMLGLKPKADAFFVMMFTLMMVAYSASSMALAIAAGQSVVSVATLLMTICFVFMMIFSGLLVNLTTIASWLSWLQYFSIPRYGFTALQHNEFLGQNFCPGLNATGNNPCNYATCTGEEYLVKQGIDLSPWGLWKNHVALACMIVIFLTIAYLKLLFLKKYS"
    },
    {
        "id": "CYP1A2",
        "sequence": "MALSQSVPFSATELLLASAIFCLVFWVLKGLRPRVPKGLKSPPEPWGWPLLGHVLTLGKNPHLALSRMSQRYGDVLQIRIGSTPVLVLSRLDTIRQALVRQGDDFKGRPDLYTSTLITDGQSLTFSTDSGPVWAARRRLAQNALNTFSIASDPASSSSCYLEEHVSKEAKALISRLQELMAGPGHFDPYNQVVVSVANVIGAMCFGQHFPESSDEMLSLVKNTHEFVETASSGNPLDFFPILRYLPNPALQRFKAFNQRFLWFLQKTVQEHYQDFDKNSVRDITGALFKHSKKGPRASGNLIPQEKIVNLVNDIFGAGFDTVTTAISWSLMYLVTKPEIQRKIQKELDTVIGRERRPRLSDRPQLPYLEAFILETFRHSSFLPFTIPHSTTRDTTLNGFYIPKKCCVFVNQWQVNHDPELWEDPSEFRPERFLTADGTAINKPLSEKMMLFGMGKRRCIGEVLAKWEIFLFLAILLQQLEFSVPPGVKVDLTPIYGLTMKHARCEHVQARLRFSIN"
    },
    {
        "id": "CYP2A6",
        "sequence": "MLASGMLLVALLVCLTVMVLMSVWQQRKSKGKLPPGPTPLPFIGNYLQLNTEQMYNSLMKISERYGPVFTIHLGPRRVVVLCGHDAVREALVDQAEEFSGRGEQATFDWVFKGYGVVFSNGERAKQLRRFSIATLRDFGVGKRGIEERIQEEAGFLIDALRGTGGANIDPTFFLSRTVSNVISSIVFGDRFDYKDKEFLSLLRMMLGIFQFTSTSTGQLYEMFSSVMKHLPGPQQQAFQLLQGLEDFIAKKVEHNQRTLDPNSPRDFIDSFLIRMQEEEKNPNTEFYLKNLVMTTLNLFIGGTETVSTTLRYGFLLLMKHPEVEAKVHEEIDRVIGKNRQPKFEDRAKMPYMEAVIHEIQRFGDVIPMSLARRVKKDTKFRDFFLPKGTEVYPMLGSVLRDPSFFSNPQDFNPQHFLNEKGQFKKSDAFVPFSIGKRNCFGEGLARMELFLFFTTVMQNFRLKSSQSPKDIDVSPKHVGFATIPRNYTMSFLPR"
    },
    {
        "id": "CYP2B6",
        "sequence": "MELSVLLFLALLTGLLLLLVQRHPNTHDRLPPGPRPLPLLGNLLQMDRRGLLKSFLRFREKYGDVFTVHLGPRPVVMLCGVEAIREALVDKAEAFSGRGKIAMVDPFFRGYGVIFANGNRWKVLRRFSVTTMRDFGMGKRSVEERIQEEAQCLIEELRKSKGALMDPTFLFQSITANIICSIVFGKRFHYQDQEFLKMLNLFYQTFSLISSVFGQLFELFSGFLKYFPGAHRQVYKNLQEINAYIGHSVEKHRETLDPSAPKDLIDTYLLHMEKEKSNAHSEFSHQNLNLNTLSLFFAGTETTSTTLRYGFLLMLKYPHVAERVYREIEQVIGPHRPPELHDRAKMPYTEAVIYEIQRFSDLLPMGVPHIVTQHTSFRGYIIPKDTEVFLILSTALHDPHYFEKPDAFNPDHFLDANGALKKTEAFIPFSLGKRICLGEGIARAELFLFFTTILQNFSMASPVAPEDIDLTPQECGVGKIPPTYQIRFLPR"
    },
    {
        "id": "CYP2C18",
        "sequence": "MDPAVALVLCLSCLFLLSLWRQSSGRGRLPSGPTPLPIIGNILQLDVKDMSKSLTNFSKVYGPVFTVYFGLKPIVVLHGYEAVKEALIDHGEEFSGRGSFPVAEKVNKGLGILFSNGKRWKEIRRFCLMTLRNFGMGKRSIEDRVQEEARCLVEELRKTNASPCDPTFILGCAPCNVICSVIFHDRFDYKDQRFLNLMEKFNENLRILSSPWIQVCNNFPALIDYLPGSHNKIAENFAYIKSYVLERIKEHQESLDMNSARDFIDCFLIKMEQEKHNQQSEFTVESLIATVTDMFGAGTETTSTTLRYGLLLLLKYPEVTAKVQEEIECVVGRNRSPCMQDRSHMPYTDAVVHEIQRYIDLLPTNLPHAVTCDVKFKNYLIPKGTTIITSLTSVLHNDKEFPNPEMFDPGHFLDKSGNFKKSDYFMPFSAGKRMCMGEGLARMELFLFLTTILQNFNLKSQVDPKDIDITPIANAFGRVPPLYQLCFIPV"
    },
    {
        "id": "CYP2C19",
        "sequence": "MDPFVVLVLCLSCLLLLSIWRQSSGRGKLPPGPTPLPVIGNILQIDIKDVSKSLTNLSKIYGPVFTLYFGLERMVVLHGYEVVKEALIDLGEEFSGRGHFPLAERANRGFGIVFSNGKRWKEIRRFSLMTLRNFGMGKRSIEDRVQEEARCLVEELRKTKASPCDPTFILGCAPCNVICSIIFQKRFDYKDQQFLNLMEKLNENIRIVSTPWIQICNNFPTIIDYFPGTHNKLLKNLAFMESDILEKVKEHQESMDINNPRDFIDCFLIKMEKEKQNQQSEFTIENLVITAADLLGAGTETTSTTLRYALLLLLKHPEVTAKVQEEIERVIGRNRSPCMQDRGHMPYTDAVVHEVQRYIDLIPTSLPHAVTCDVKFRNYLIPKGTTILTSLTSVLHDNKEFPNPEMFDPRHFLDEGGNFKKSNYFMPFSAGKRICVGEGLARMELFLFLTFILQNFNLKSLIDPKDLDTTPVVNGFASVPPFYQLCFIPV"
    },
    {
        "id": "CYP2C8",
        "sequence": "MEPFVVLVLCLSFMLLFSLWRQSCRRRKLPPGPTPLPIIGNMLQIDVKDICKSFTNFSKVYGPVFTVYFGMNPIVVFHGYEAVKEALIDNGEEFSGRGNSPISQRITKGLGIISSNGKRWKEIRRFSLTTLRNFGMGKRSIEDRVQEEAHCLVEELRKTKASPCDPTFILGCAPCNVICSVVFQKRFDYKDQNFLTLMKRFNENFRILNSPWIQVCNNFPLLIDCFPGTHNKVLKNVALTRSYIREKVKEHQASLDVNNPRDFIDCFLIKMEQEKDNQKSEFNIENLVGTVADLFVAGTETTSTTLRYGLLLLLKHPEVTAKVQEEIDHVIGRHRSPCMQDRSHMPYTDAVVHEIQRYSDLVPTGVPHAVTTDTKFRNYLIPKGTTIMALLTSVLHDDKEFPNPNIFDPGHFLDKNGNFKKSDYFMPFSAGKRICAGEGLARMELFLFLTTILQNFNLKSVDDLKNLNTTAVTKGIVSLPPSYQICFIPV"
    },
    {
        "id": "CYP2C9",
        "sequence": "MDSLVVLVLCLSCLLLLSLWRQSSGRGKLPPGPTPLPVIGNILQIGIKDISKSLTNLSKVYGPVFTLYFGLKPIVVLHGYEAVKEALIDLGEEFSGRGIFPLAERANRGFGIVFSNGKKWKEIRRFSLMTLRNFGMGKRSIEDRVQEEARCLVEELRKTKASPCDPTFILGCAPCNVICSIIFHKRFDYKDQQFLNLMEKLNENIKILSSPWIQICNNFSPIIDYFPGTHNKLLKNVAFMKSYILEKVKEHQESMDMNNPQDFIDCFLMKMEKEKHNQPSEFTIESLENTAVDLFGAGTETTSTTLRYALLLLLKHPEVTAKVQEEIERVIGRNRSPCMQDRSHMPYTDAVVHEVQRYIDLLPTSLPHAVTCDIKFRNYLIPKGTTILISLTSVLHDNKEFPNPEMFDPHHFLDEGGNFKKSKYFMPFSAGKRICVGEALAGMELFLFLTSILQNFNLKSLVDPKNLDTTPVVNGFASVPPFYQLCFIPV"
    },
    {
        "id": "CYP2D6",
        "sequence": "MGLEALVPLAVIVAIFLLLVDLMHRRQRWAARYPPGPLPLPGLGNLLHVDFQNTPYCFDQLRRRFGDVFSLQLAWTPVVVLNGLAAVREALVTHGEDTADRPPVPITQILGFGPRSQGVFLARYGPAWREQRRFSVSTLRNLGLGKKSLEQWVTEEAACLCAAFANHSGRPFRPNGLLDKAVSNVIASLTCGRRFEYDDPRFLRLLDLAQEGLKEESGFLREVLNAVPVLLHIPALAGKVLRFQKAFLTQLDELLTEHRMTWDPAQPPRDLTEAFLAEMEKAKGNPESSFNDENLRIVVADLFSAGMVTTSTTLAWGLLLMILHPDVQRRVQQEIDDVIGQVRRPEMGDQAHMPYTTAVIHEVQRFGDIVPLGVTHMTSRDIEVQGFRIPKGTTLITNLSSVLKDEAVWEKPFRFHPEHFLDAQGHFVKPEAFLPFSAGRRACLGEPLARMELFLFFTSLLQHFSFSVPTGQPRPSHHGVFAFLVSPSPYELCAVPR"
    },
    {
        "id": "CYP2E1",
        "sequence": "MSALGVTVALLVWAAFLLLVSMWRQVHSSWNLPPGPFPLPIIGNLFQLELKNIPKSFTRLAQRFGPVFTLYVGSQRMVVMHGYKAVKEALLDYKDEFSGRGDLPAFHAHRDRGIIFNNGPTWKDIRRFSLTTLRNYGMGKQGNESRIQREAHFLLEALRKTQGQPFDPTFLIGCAPCNVIADILFRKHFDYNDEKFLRLMYLFNENFHLLSTPWLQLYNNFPSFLHYLPGSHRKVIKNVAEVKEYVSERVKEHHQSLDPNCPRDLTDCLLVEMEKEKHSAERLYTMDGITVTVADLFFAGTETTSTTLRYGLLILMKYPEIEEKLHEEIDRVIGPSRIPAIKDRQEMPYMDAVVHEIQRFITLVPSNLPHEATRDTIFRGYLIPKGTVVVPTLDSVLYDNQEFPDPEKFKPEHFLNENGKFKYSDYFKPFSTGKRVCAGEGLARMELFLLLCAILQHFNLKPLVDPKDIDLSPIHIGFGCIPPRYKLCVIPRS"
    },
    {
        "id": "CYP3A4",
        "sequence": "MALIPDLAMETWLLLAVSLVLLYLYGTHSHGLFKKLGIPGPTPLPFLGNILSYHKGFCMFDMECHKKYGKVWGFYDGQQPVLAITDPDMIKTVLVKECYSVFTNRRPFGPVGFMKSAISIAEDEEWKRLRSLLSPTFTSGKLKEMVPIIAQYGDVLVRNLRREAETGKPVTLKDVFGAYSMDVITSTSFGVNIDSLNNPQDPFVENTKKLLRFDFLDPFFLSITVFPFLIPILEVLNICVFPREVTNFLRKSVKRMKESRLEDTQKHRVDFLQLMIDSQNSKETESHKALSDLELVAQSIIFIFAGYETTSSVLSFIMYELATHPDVQQKLQEEIDAVLPNKAPPTYDTVLQMEYLDMVVNETLRLFPIAMRLERVCKKDVEINGMFIPKGVVVMIPSYALHRDPKYWTEPEKFLPERFSKKNKDNIDPYIYTPFGSGPRNCIGMRFALMNMKLALIRVLQNFSFKPCKETQIPLKLSLGGLLQPEKPVVLKVESRDGTVSGA"
    },
    {
        "id": "CYP3A5",
        "sequence": "MDLIPNLAVETWLLLAVSLVLLYLYGTRTHGLFKRLGIPGPTPLPLLGNVLSYRQGLWKFDTECYKKYGKMWGTYEGQLPVLAITDPDVIRTVLVKECYSVFTNRRSLGPVGFMKSAISLAEDEEWKRIRSLLSPTFTSGKLKEMFPIIAQYGDVLVRNLRREAEKGKPVTLKDIFGAYSMDVITGTSFGVNIDSLNNPQDPFVESTKKFLKFGFLDPLFLSIILFPFLTPVFEALNVSLFPKDTINFLSKSVNRMKKSRLNDKQKHRLDFLQLMIDSQNSKETESHKALSDLELAAQSIIFIFAGYETTSSVLSFTLYELATHPDVQQKLQKEIDAVLPNKAPPTYDAVVQMEYLDMVVNETLRLFPVAIRLERTCKKDVEINGVFIPKGSMVVIPTYALHHDPKYWTEPEEFRPERFSKKKDSIDPYIYTPFGTGPRNCIGMRFALMNMKLALIRVLQNFSFKPCKETQIPLKLDTQGLLQPEKPIVLKVDSRDGTLSGE"
    },
    {
        "id": "CYP4A11",
        "sequence": "MSVSVLSPSRLLGDVSGILQAASLLILLLLLIKAVQLYLHRQWLLKALQQFPCPPSHWLFGHIQELQQDQELQRIQKWVETFPSACPHWLWGGKVRVQLYDPDYMKVILGRSDPKSHGSYRFLAPWIGYGLLLLNGQTWFQHRRMLTPAFHYDILKPYVGLMADSVRVMLDKWEELLGQDSPLEVFQHVSLMTLDTIMKCAFSHQGSIQVDRNSQSYIQAISDLNNLVFSRVRNAFHQNDTIYSLTSAGRWTHRACQLAHQHTDQVIQLRKAQLQKEGELEKIKRKRHLDFLDILLLAKMENGSILSDKDLRAEVDTFMFEGHDTTASGISWILYALATHPKHQERCREEIHSLLGDGASITWNHLDQMPYTTMCIKEALRLYPPVPGIGRELSTPVTFPDGRSLPKGIMVLLSIYGLHHNPKVWPNPEVFDPFRFAPGSAQHSHAFLPFSGGSRNCIGKQFAMNELKVATALTLLRFELLPDPTRIPIPIARLVLKSKNGIHLRLRRLPNPCEDKDQL"
    },
    {
        "id": "NTCP",
        "sequence": "MEAHNASAPFNFTLPPNFGKRPTDLALSVILVFMLFFIMLSLGCTMEFSKIKAHLWKPKGLAIALVAQYGIMPLTAFVLGKVFRLKNIEALAILVCGCSPGGNLSNVFSLAMKGDMNLSIVMTTCSTFCALGMMPLLLYIYSRGIYDGDLKDKVPYKGIVISLVLVLIPCTIGIVLKSKRPQYMRYVIKGGMIIILLCSVAVTVLSAINVGKSIMFAMTPLLIATSSLMPFIGFLLGYVLSALFCLNGRCRRTVSMETGCQNVQLCSTILNVAFPPEVIGPLFFFPLLYMIFQLGEGLLLIAIFWCYEKFKTPKDKTKMIYTAATTEETIPGALGNGTYKGEDCSPCTA"
    },
    {
        "id": "OAT4",
        "sequence": "MAFSKLLEQAGGVGLFQTLQVLTFILPCLMIPSQMLLENFSAAIPGHRCWTHMLDNGSAVSTNMTPKALLTISIPPGPNQGPHQCRRFRQPQWQLLDPNATATSWSEADTEPCVDGWVYDRSVFTSTIVAKWDLVCSSQGLKPLSQSIFMSGILVGSFIWGLLSYRFGRKPMLSWCCLQLAVAGTSTIFAPTFVIYCGLRFVAAFGMAGIFLSSLTLMVEWTTTSRRAVTMTVVGCAFSAGQAALGGLAFALRDWRTLQLAASVPFFAISLISWWLPESARWLIIKGKPDQALQELRKVARINGHKEAKNLTIEVLMSSVKEEVASAKEPRSVLDLFCVPVLRWRSCAMLVVNFSLLISYYGLVFDLQSLGRDIFLLQALFGAVDFLGRATTALLLSFLGRRTIQAGSQAMAGLAILANMLVPQDLQTLRVVFAVLGKGCFGISLTCLTIYKAELFPTPVRMTADGILHTVGRLGAMMGPLILMSRQALPLLPPLLYGVISIASSLVVLFFLPETQGLPLPDTIQDLESQKSTAAQGNRQEAVTVESTSL"
    },
    {
        "id": "SLC22A1",
        "sequence": "MPTVDDILEQVGESGWFQKQAFLILCLLSAAFAPICVGIVFLGFTPDHHCQSPGVAELSQRCGWSPAEELNYTVPGLGPAGEAFLGQCRRYEVDWNQSALSCVDPLASLATNRSHLPLGPCQDGWVYDTPGSSIVTEFNLVCADSWKLDLFQSCLNAGFLFGSLGVGYFADRFGRKLCLLGTVLVNAVSGVLMAFSPNYMSMLLFRLLQGLVSKGNWMAGYTLITEFVGSGSRRTVAIMYQMAFTVGLVALTGLAYALPHWRWLQLAVSLPTFLFLLYYWCVPESPRWLLSQKRNTEAIKIMDHIAQKNGKLPPADLKMLSLEEDVTEKLSPSFADLFRTPRLRKRTFILMYLWFTDSVLYQGLILHMGATSGNLYLDFLYSALVEIPGAFIALITIDRVGRIYPMAMSNLLAGAACLVMIFISPDLHWLNIIIMCVGRMGITIAIQMICLVNAELYPTFVRNLGVMVCSSLCDIGGIITPFIVFRLREVWQALPLILFAVLGLLAAGVTLLLPETKGVALPETMKDAENLGRKAKPKENTIYLKVQTSEPSGT"
    },
    {
        "id": "SLC22A2",
        "sequence": "MPTTVDDVLEHGGEFHFFQKQMFFLLALLSATFAPIYVGIVFLGFTPDHRCRSPGVAELSLRCGWSPAEELNYTVPGPGPAGEASPRQCRRYEVDWNQSTFDCVDPLASLDTNRSRLPLGPCRDGWVYETPGSSIVTEFNLVCANSWMLDLFQSSVNVGFFIGSMSIGYIADRFGRKLCLLTTVLINAAAGVLMAISPTYTWMLIFRLIQGLVSKAGWLIGYILITEFVGRRYRRTVGIFYQVAYTVGLLVLAGVAYALPHWRWLQFTVSLPNFFFLLYYWCIPESPRWLISQNKNAEAMRIIKHIAKKNGKSLPASLQRLRLEEETGKKLNPSFLDLVRTPQIRKHTMILMYNWFTSSVLYQGLIMHMGLAGDNIYLDFFYSALVEFPAAFMIILTIDRIGRRYPWAASNMVAGAACLASVFIPGDLQWLKIIISCLGRMGITMAYEIVCLVNAELYPTFIRNLGVHICSSMCDIGGIITPFLVYRLTNIWLELPLMVFGVLGLVAGGLVLLLPETKGKALPETIEEAENMQRPRKNKEKMIYLQVQKLDIPLN"
    },
    {
        "id": "SLC22A4",
        "sequence": "MRDYDEVIAFLGEWGPFQRLIFFLLSASIIPNGFNGMSVVFLAGTPEHRCRVPDAANLSSAWRNNSVPLRLRDGREVPHSCSRYRLATIANFSALGLEPGRDVDLGQLEQESCLDGWEFSQDVYLSTVVTEWNLVCEDNWKVPLTTSLFFVGVLLGSFVSGQLSDRFGRKNVLFATMAVQTGFSFLQIFSISWEMFTVLFVIVGMGQISNYVVAFILGTEILGKSVRIIFSTLGVCTFFAVGYMLLPLFAYFIRDWRMLLLALTVPGVLCVPLWWFIPESPRWLISQRRFREAEDIIQKAAKMNNIAVPAVIFDSVEELNPLKQQKAFILDLFRTRNIAIMTIMSLLLWMLTSVGYFALSLDAPNLHGDAYLNCFLSALIEIPAYITAWLLLRTLPRRYIIAAVLFWGGGVLLFIQLVPVDYYFLSIGLVMLGKFGITSAFSMLYVFTAELYPTLVRNMAVGVTSTASRVGSIIAPYFVYLGAYNRMLPYIVMGSLTVLIGILTLFFPESLGMTLPETLEQMQKVKWFRSGKKTRDSMETEENPKVLITAF"
    },
    {
        "id": "SLC22A5",
        "sequence": "MRDYDEVTAFLGEWGPFQRLIFFLLSASIIPNGFTGLSSVFLIATPEHRCRVPDAANLSSAWRNHTVPLRLRDGREVPHSCRRYRLATIANFSALGLEPGRDVDLGQLEQESCLDGWEFSQDVYLSTIVTEWNLVCEDDWKAPLTISLFFVGVLLGSFISGQLSDRFGRKNVLFVTMGMQTGFSFLQIFSKNFEMFVVLFVLVGMGQISNYVAAFVLGTEILGKSVRIIFSTLGVCIFYAFGYMVLPLFAYFIRDWRMLLVALTMPGVLCVALWWFIPESPRWLISQGRFEEAEVIIRKAAKANGIVVPSTIFDPSELQDLSSKKQQSHNILDLLRTWNIRMVTIMSIMLWMTISVGYFGLSLDTPNLHGDIFVNCFLSAMVEVPAYVLAWLLLQYLPRRYSMATALFLGGSVLLFMQLVPPDLYYLATVLVMVGKFGVTAAFSMVYVYTAELYPTVVRNMGVGVSSTASRLGSILSPYFVYLGAYDRFLPYILMGSLTILTAILTLFLPESFGTPLPDTIDQMLRVKGMKHRKTPSHTRMLKDGQERPTILKSTAF"
    },
    {
        "id": "SLC22A6",
        "sequence": "MAFNDLLQQVGGVGRFQQIQVTLVVLPLLLMASHNTLQNFTAAIPTHHCRPPADANLSKNGGLEVWLPRDRQGQPESCLRFTSPQWGLPFLNGTEANGTGATEPCTDGWIYDNSTFPSTIVTEWDLVCSHRALRQLAQSLYMVGVLLGAMVFGYLADRLGRRKVLILNYLQTAVSGTCAAFAPNFPIYCAFRLLSGMALAGISLNCMTLNVEWMPIHTRACVGTLIGYVYSLGQFLLAGVAYAVPHWRHLQLLVSAPFFAFFIYSWFFIESARWHSSSGRLDLTLRALQRVARINGKREEGAKLSMEVLRASLQKELTMGKGQASAMELLRCPTLRHLFLCLSMLWFATSFAYYGLVMDLQGFGVSIYLIQVIFGAVDLPAKLVGFLVINSLGRRPAQMAALLLAGICILLNGVIPQDQSIVRTSLAVLGKGCLAASFNCIFLYTGELYPTMIRQTGMGMGSTMARVGSIVSPLVSMTAELYPSMPLFIYGAVPVAASAVTVLLPETLGQPLPDTVQDLESRWAPTQKEAGIYPRKGKQTRQQQEHQKYMVPLQASAQEKNGL"
    },
    {
        "id": "SLC22A7",
        "sequence": "MGFEELLEQVGGFGPFQLRNVALLALPRVLLPLHFLLPIFLAAVPAHRCALPGAPANFSHQDVWLEAHLPREPDGTLSSCLRFAYPQALPNTTLGEERQSRGELEDEPATVPCSQGWEYDHSEFSSTIATESQWDLVCEQKGLNRAASTFFFAGVLVGAVAFGYLSDRFGRRRLLLVAYVSTLVLGLASAASVSYVMFAITRTLTGSALAGFTIIVMPLELEWLDVEHRTVAGVLSSTFWTGGVMLLALVGYLIRDWRWLLLAVTLPCAPGILSLWWVPESARWLLTQGHVKEAHRYLLHCARLNGRPVCEDSFSQEAVSKVAAGERVVRRPSYLDLFRTPRLRHISLCCVVVWFGVNFSYYGLSLDVSGLGLNVYQTQLLFGAVELPSKLLVYLSVRYAGRRLTQAGTLLGTALAFGTRLLVSSDMKSWSTVLAVMGKAFSEAAFTTAYLFTSELYPTVLRQTGMGLTALVGRLGGSLAPLAALLDGVWLSLPKLTYGGIALLAAGTALLLPETRQAQLPETIQDVERKSAPTSLQEEEMPMKQVQN"
    },
    {
        "id": "SLC22A8",
        "sequence": "MTFSEILDRVGSMGHFQFLHVAILGLPILNMANHNLLQIFTAATPVHHCRPPHNASTGPWVLPMGPNGKPERCLRFVHPPNASLPNDTQRAMEPCLDGWVYNSTKDSIVTEWDLVCNSNKLKEMAQSIFMAGILIGGLVLGDLSDRFGRRPILTCSYLLLAASGSGAAFSPTFPIYMVFRFLCGFGISGITLSTVILNVEWVPTRMRAIMSTALGYCYTFGQFILPGLAYAIPQWRWLQLTVSIPFFVFFLSSWWTPESIRWLVLSGKSSKALKILRRVAVFNGKKEEGERLSLEELKLNLQKEISLAKAKYTASDLFRIPMLRRMTFCLSLAWFATGFAYYSLAMGVEEFGVNLYILQIIFGGVDVPAKFITILSLSYLGRHTTQAAALLLAGGAILALTFVPLDLQTVRTVLAVFGKGCLSSSFSCLFLYTSELYPTVIRQTGMGVSNLWTRVGSMVSPLVKITGEVQPFIPNIIYGITALLGGSAALFLPETLNQPLPETIEDLENWSLRAKKPKQEPEVEKASQRIPLQPHGPGLGSS"
    },
    {
        "id": "SLC22A9",
        "sequence": "MAFQDLLGHAGDLWRFQILQTVFLSIFAVATYLHFMLENFTAFIPGHRCWVHILDNDTVSDNDTGALSQDALLRISIPLDSNMRPEKCRRFVHPQWQLLHLNGTFPNTSDADMEPCVDGWVYDRISFSSTIVTEWDLVCDSQSLTSVAKFVFMAGMMVGGILGGHLSDRFGRRFVLRWCYLQVAIVGTCAALAPTFLIYCSLRFLSGIAAMSLITNTIMLIAEWATHRFQAMGITLGMCPSGIAFMTLAGLAFAIRDWHILQLVVSVPYFVIFLTSSWLLESARWLIINNKPEEGLKELRKAAHRSGMKNARDTLTLEILKSTMKKELEAAQKKKPSLCEMLHMPNICKRISLLSFTRFANFMAYFGLNLHVQHLGNNVFLLQTLFGAVILLANCVAPWALKYMNRRASQMLLMFLLAICLLAIIFVPQEMQTLREVLATLGLGASALANTLAFAHGNEVIPTIIRARAMGINATFANIAGALAPLMMILSVYSPPLPWIIYGVFPFISGFAFLLLPETRNKPLFDTIQDEKNERKDPREPKQEDPRVEVTQF"
    },
    {
        "id": "SLC47A1",
        "sequence": "MEAPEEPAPVRGGPEATLEVRGSRCLRLSAFREELRALLVLAGPAFLVQLMVFLISFISSVFCGHLGKLELDAVTLAIAVINVTGVSVGFGLSSACDTLISQTYGSQNLKHVGVILQRSALVLLLCCFPCWALFLNTQHILLLFRQDPDVSRLTQTYVTIFIPALPATFLYMLQVKYLLNQGIVLPQIVTGVAANLVNALANYLFLHQLHLGVIGSALANLISQYTLALLLFLYILGKKLHQATWGGWSLECLQDWASFLRLAIPSMLMLCMEWWAYEVGSFLSGILGMVELGAQSIVYELAIIVYMVPAGFSVAASVRVGNALGAGDMEQARKSSTVSLLITVLFAVAFSVLLLSCKDHVGYIFTTDRDIINLVAQVVPIYAVSHLFEALACTSGGVLRGSGNQKVGAIVNTIGYYVVGLPIGIALMFATTLGVMGLWSGIIICTVFQAVCFLGFIIQLNWKKACQQAQVHANLKVNNVPRSGNSALPQDPLHPGCPENLEGILTNDVGKTGEPQSDQQMRQEEPLPEHPQDGAKLSRKQLVLRRGLLLLGVFLILLVGILVRFYVRIQ"
    },
    {
        "id": "SLC47A2",
        "sequence": "MDSLQDTVALDHGGCCPALSRLVPRGFGTEMWTLFALSGPLFLFQVLTFMIYIVSTVFCGHLGKVELASVTLAVAFVNVCGVSVGVGLSSACDTLMSQSFGSPNKKHVGVILQRGALVLLLCCLPCWALFLNTQHILLLFRQDPDVSRLTQDYVMIFIPGLPVIFLYNLLAKYLQNQGWLKGQEEESPFQTPGLSILHPSHSHLSRASFHLFQKITWPQVLSGVVGNCVNGVANYALVSVLNLGVRGSAYANIISQFAQTVFLLLYIVLKKLHLETWAGWSSQCLQDWGPFFSLAVPSMLMICVEWWAYEIGSFLMGLLSVVDLSAQAVIYEVATVTYMIPLGLSIGVCVRVGMALGAADTVQAKRSAVSGVLSIVGISLVLGTLISILKNQLGHIFTNDEDVIALVSQVLPVYSVFHVFEAICCVYGGVLRGTGKQAFGAAVNAITYYIIGLPLGILLTFVVRMRIMGLWLGMLACVFLATAAFVAYTARLDWKLAAEEAKKHSGRQQQQRAESTATRPGPEKAVLSSVATGSSPGITLTTYSRSECHVDFFRTPEEAHALSAPTSRLSVKQLVIRRGAALGAASATLMVGLTVRILATRH"
    },
    {
        "id": "SLCO1A2",
        "sequence": "MGETEKRIETHRIRCLSKLKMFLLAITCAFVSKTLSGSYMNSMLTQIERQFNIPTSLVGFINGSFEIGNLLLIIFVSYFGTKLHRPIMIGIGCVVMGLGCFLKSLPHFLMNQYEYESTVSVSGNLSSNSFLCMENGTQILRPTQDPSECTKEVKSLMWVYVLVGNIVRGMGETPILPLGISYIEDFAKFENSPLYIGLVETGAIIGPLIGLLLASFCANVYVDTGFVNTDDLIITPTDTRWVGAWWFGFLICAGVNVLTAIPFFFLPNTLPKEGLETNADIIKNENEDKQKEEVKKEKYGITKDFLPFMKSLSCNPIYMLFILVSVIQFNAFVNMISFMPKYLEQQYGISSSDAIFLMGIYNLPPICIGYIIGGLIMKKFKITVKQAAHIGCWLSLLEYLLYFLSFLMTCENSSVVGINTSYEGIPQDLYVENDIFADCNVDCNCPSKIWDPVCGNNGLSYLSACLAGCETSIGTGINMVFQNCSCIQTSGNSSAVLGLCDKGPDCSLMLQYFLILSAMSSFIYSLAAIPGYMVLLRCMKSEEKSLGVGLHTFCTRVFAGIPAPIYFGALMDSTCLHWGTLKCGESGACRIYDSTTFRYIYLGLPAALRGSSFVPALIILILLRKCHLPGENASSGTELIETKVKGKENECKDIYQKSTVLKDDELKTKL"
    },
    {
        "id": "SLCO1B1",
        "sequence": "MDQNQHLNKTAEAQPSENKKTRYCNGLKMFLAALSLSFIAKTLGAIIMKSSIIHIERRFEISSSLVGFIDGSFEIGNLLVIVFVSYFGSKLHRPKLIGIGCFIMGIGGVLTALPHFFMGYYRYSKETNINSSENSTSTLSTCLINQILSLNRASPEIVGKGCLKESGSYMWIYVFMGNMLRGIGETPIVPLGLSYIDDFAKEGHSSLYLGILNAIAMIGPIIGFTLGSLFSKMYVDIGYVDLSTIRITPTDSRWVGAWWLNFLVSGLFSIISSIPFFFLPQTPNKPQKERKASLSLHVLETNDEKDQTANLTNQGKNITKNVTGFFQSFKSILTNPLYVMFVLLTLLQVSSYIGAFTYVFKYVEQQYGQPSSKANILLGVITIPIFASGMFLGGYIIKKFKLNTVGIAKFSCFTAVMSLSFYLLYFFILCENKSVAGLTMTYDGNNPVTSHRDVPLSYCNSDCNCDESQWEPVCGNNGITYISPCLAGCKSSSGNKKPIVFYNCSCLEVTGLQNRNYSAHLGECPRDDACTRKFYFFVAIQVLNLFFSALGGTSHVMLIVKIVQPELKSLALGFHSMVIRALGGILAPIYFGALIDTTCIKWSTNNCGTRGSCRTYNSTSFSRVYLGLSSMLRVSSLVLYIILIYAMKKKYQEKDINASENGSVMDEANLESLNKNKHFVPSAGADSETHC"
    },
    {
        "id": "SLCO1B3",
        "sequence": "MDQHQHLNKTAESASSEKKKTRRCNGFKMFLAALSFSYIAKALGGIIMKISITQIERRFDISSSLAGLIDGSFEIGNLLVIVFVSYFGSKLHRPKLIGIGCLLMGTGSILTSLPHFFMGYYRYSKETNIDPSENSTSNLPNCLINQMLSLNRTPSEIIERGCVKESGSHMWIYVFMGNMLRGIGETPIVPLGISYIDDFAKEGHSSLYLGTVNVMGMTGLVFAFMLGSLFAKMYVDIGYVDLSTIRITPKDSRWVGAWWLGFLVSGIVSIISSIPFFFLPLNPNKPQKERKVSLFLHVLKTNDKRNQIANLTNRRKYITKNVTGFFQSLKSILTNPLYVIFVIFTLLHMSSYIASLTYIIKMVEQQYGWSASKTNFLLGVLALPAVAIGMFSGGYIIKKFKLSLVGLAKLAFCSATVHLLSQVLYFFLICESKSVAGLTLTYDGNSPVRSHVDVPLSYCNSECNCDESQWEPVCGNNGITYLSPCLAGCKSSSGNKEPIVFYNCSCVEVIGLQNKNYSAHLGECPRDDACTRKSYVYFVIQVLDAFLCAVGLTSYSVLVIRIVQPELKALAIGFHSMIMRSLGGILVPIYFGALIDTTCMKWSTNSCGARGACRIYNSTYLGRAFFGLKVALIFPVLVLLTVFIFVVRKKSHGKDTKVLENERQVMDEANLEFLNDSEHFVPSAEEQ"
    },
    {
        "id": "SLCO2B1",
        "sequence": "MGPRIGPAGEVPQVPDKETKATMGTENTPGGKASPDPQDVRPSVFHNIKLFVLCHSLLQLAQLMISGYLKSSISTVEKRFGLSSQTSGLLASFNEVGNTALIVFVSYFGSRVHRPRMIGYGAILVALAGLLMTLPHFISEPYRYDNTSPEDMPQDFKASLCLPTTSAPASAPSNGNCSSYTETQHLSVVGIMFVAQTLLGVGGVPIQPFGISYIDDFAHNSNSPLYLGILFAVTMMGPGLAFGLGSLMLRLYVDINQMPEGGISLTIKDPRWVGAWWLGFLIAAGAVALAAIPYFFFPKEMPKEKRELQFRRKVLAVTDSPARKGKDSPSKQSPGESTKKQDGLVQIAPNLTVIQFIKVFPRVLLQTLRHPIFLLVVLSQVCLSSMAAGMATFLPKFLERQFSITASYANLLIGCLSFPSVIVGIVVGGVLVKRLHLGPVGCGALCLLGMLLCLFFSLPLFFIGCSSHQIAGITHQTSAHPGLELSPSCMEACSCPLDGFNPVCDPSTRVEYITPCHAGCSSWVVQDALDNSQVFYTNCSCVVEGNPVLAGSCDSTCSHLVVPFLLLVSLGSALACLTHTPSFMLILRGVKKEDKTLAVGIQFMFLRILAWMPSPVIHGSAIDTTCVHWALSCGRRAVCRYYNNDLLRNRFIGLQFFFKTGSVICFALVLAVLRQQDKEARTKESRSSPAVEQQLLVSGPGKKPEDSRV"
    },
    {
        "id": "SLCO4C1",
        "sequence": "MKSAKGIENLAFVPSSPDILRRLSASPSQIEVSALSSDPQRENSQPQELQKPQEPQKSPEPSLPSAPPNVSEEKLRSLSLSEFEEGSYGWRNFHPQCLQRCNTPGGFLLHYCLLAVTQGIVVNGLVNISISTVEKRYEMKSSLTGLISSSYDISFCLLSLFVSFFGERGHKPRWLAFAAFMIGLGALVFSLPQFFSGEYKLGSLFEDTCVTTRNSTSCTSSTSSLSNYLYVFILGQLLLGAGGTPLYTLGTAFLDDSVPTHKSSLYIGTGYAMSILGPAIGYVLGGQLLTIYIDVAMGESTDVTEDDPRWLGAWWIGFLLSWIFAWSLIIPFSCFPKHLPGTAEIQAGKTSQAHQSNSNADVKFGKSIKDFPAALKNLMKNAVFMCLVLSTSSEALITTGFATFLPKFIENQFGLTSSFAATLGGAVLIPGAALGQILGGFLVSKFRMTCKNTMKFALFTSGVALTLSFVFMYAKCENEPFAGVSESYNGTGELGNLIAPCNANCNCSRSYYYPVCGDGVQYFSPCFAGCSNPVAHRKPKVYYNCSCIERKTEITSTAETFGFEAKAGKCETHCAKLPIFLCIFFIVIIFTFMAGTPITVSILRCVNHRQRSLALGIQFMVLRLLGTIPGPIIFGFTIDSTCILWDINDCGIKGACWIYDNIKMAHMLVAISVTCKVITMFFNGFAIFLYKPPPSATDVSFHKENAVVTNVLAEQDLNKIVKEG"
    },
    {
        "id": "TRPV4",
        "sequence": "MADSSEGPRAGPGEVAELPGDESGTPGGEAFPLSSLANLFEGEDGSLSPSPADASRPAGPGDGRPNLRMKFQGAFRKGVPNPIDLLESTLYESSVVPGPKKAPMDSLFDYGTYRHHSSDNKRWRKKIIEKQPQSPKAPAPQPPPILKVFNRPILFDIVSRGSTADLDGLLPFLLTHKKRLTDEEFREPSTGKTCLPKALLNLSNGRNDTIPVLLDIAERTGNMREFINSPFRDIYYRGQTALHIAIERRCKHYVELLVAQGADVHAQARGRFFQPKDEGGYFYFGELPLSLAACTNQPHIVNYLTENPHKKADMRRQDSRGNTVLHALVAIADNTRENTKFVTKMYDLLLLKCARLFPDSNLEAVLNNDGLSPLMMAAKTGKIGIFQHIIRREVTDEDTRHLSRKFKDWAYGPVYSSLYDLSSLDTCGEEASVLEILVYNSKIENRHEMLAVEPINELLRDKWRKFGAVSFYINVVSYLCAMVIFTLTAYYQPLEGTPPYPYRTTVDYLRLAGEVITLFTGVLFFFTNIKDLFMKKCPGVNSLFIDGSFQLLYFIYSVLVIVSAALYLAGIEAYLAVMVFALVLGWMNALYFTRGLKLTGTYSIMIQKILFKDLFRFLLVYLLFMIGYASALVSLLNPCANMKVCNEDQTNCTVPTYPSCRDSETFSTFLLDLFKLTIGMGDLEMLSSTKYPVVFIILLVTYIILTFVLLLNMLIALMGETVGQVSKESKHIWKLQWATTILDIERSFPVFLRKAFRSGEMVTVGKSSDGTPDRRWCFRVDEVNWSHWNQNLGIINEDPGKNETYQYYGFSHTVGRLRRDRWSSVVPRVVELNKNSNPDEVVVPLDSMGNPRCDGHQQGYPRKWRTDDAPL"
    },
    {
        "id": "UGT1A1",
        "sequence": "MAVESQGGRPLVLGLLLCVLGPVVSHAGKILLIPVDGSHWLSMLGAIQQLQQRGHEIVVLAPDASLYIRDGAFYTLKTYPVPFQREDVKESFVSLGHNVFENDSFLQRVIKTYKKIKKDSAMLLSGCSHLLHNKELMASLAESSFDVMLTDPFLPCSPIVAQYLSLPTVFFLHALPCSLEFEATQCPNPFSYVPRPLSSHSDHMTFLQRVKNMLIAFSQNFLCDVVYSPYATLASEFLQREVTVQDLLSSASVWLFRSDFVKDYPRPIMPNMVFVGGINCLHQNPLSQEFEAYINASGEHGIVVFSLGSMVSEIPEKKAMAIADALGKIPQTVLWRYTGTRPSNLANNTILVKWLPQNDLLGHPMTRAFITHAGSHGVYESICNGVPMVMMPLFGDQMDNAKRMETKGAGVTLNVLEMTSEDLENALKAVINDKSYKENIMRLSSLHKDRPVEPLDLAVFWVEFVMRHKGAPHLRPAAHDLTWYQYHSLDVIGFLLAVVLTVAFITFKCCAYGYRKCLGKKGRVKKAHKSKTH"
    },
    {
        "id": "UGT1A4",
        "sequence": "MARGLQVPLPRLATGLLLLLSVQPWAESGKVLVVPTDGSPWLSMREALRELHARGHQAVVLTPEVNMHIKEEKFFTLTAYAVPWTQKEFDRVTLGYTQGFFETEHLLKRYSRSMAIMNNVSLALHRCCVELLHNEALIRHLNATSFDVVLTDPVNLCGAVLAKYLSIPAVFFWRYIPCDLDFKGTQCPNPSSYIPKLLTTNSDHMTFLQRVKNMLYPLALSYICHTFSAPYASLASELFQREVSVVDLVSYASVWLFRGDFVMDYPRPIMPNMVFIGGINCANGKPLSQEFEAYINASGEHGIVVFSLGSMVSEIPEKKAMAIADALGKIPQTVLWRYTGTRPSNLANNTILVKWLPQNDLLGHPMTRAFITHAGSHGVYESICNGVPMVMMPLFGDQMDNAKRMETKGAGVTLNVLEMTSEDLENALKAVINDKSYKENIMRLSSLHKDRPVEPLDLAVFWVEFVMRHKGAPHLRPAAHDLTWYQYHSLDVIGFLLAVVLTVAFITFKCCAYGYRKCLGKKGRVKKAHKSKTH"
    },
    {
        "id": "UGT1A6",
        "sequence": "MACLLRSFQRISAGVFFLALWGMVVGDKLLVVPQDGSHWLSMKDIVEVLSDRGHEIVVVVPEVNLLLKESKYYTRKIYPVPYDQEELKNRYQSFGNNHFAERSFLTAPQTEYRNNMIVIGLYFINCQSLLQDRDTLNFFKESKFDALFTDPALPCGVILAEYLGLPSVYLFRGFPCSLEHTFSRSPDPVSYIPRCYTKFSDHMTFSQRVANFLVNLLEPYLFYCLFSKYEELASAVLKRDVDIITLYQKVSVWLLRYDFVLEYPRPVMPNMVFIGGINCKKRKDLSQEFEAYINASGEHGIVVFSLGSMVSEIPEKKAMAIADALGKIPQTVLWRYTGTRPSNLANNTILVKWLPQNDLLGHPMTRAFITHAGSHGVYESICNGVPMVMMPLFGDQMDNAKRMETKGAGVTLNVLEMTSEDLENALKAVINDKSYKENIMRLSSLHKDRPVEPLDLAVFWVEFVMRHKGAPHLRPAAHDLTWYQYHSLDVIGFLLAVVLTVAFITFKCCAYGYRKCLGKKGRVKKAHKSKTH"
    },
    {
        "id": "UGT1A9",
        "sequence": "MACTGWTSPLPLCVCLLLTCGFAEAGKLLVVPMDGSHWFTMRSVVEKLILRGHEVVVVMPEVSWQLGRSLNCTVKTYSTSYTLEDLDREFKAFAHAQWKAQVRSIYSLLMGSYNDIFDLFFSNCRSLFKDKKLVEYLKESSFDAVFLDPFDNCGLIVAKYFSLPSVVFARGILCHYLEEGAQCPAPLSYVPRILLGFSDAMTFKERVRNHIMHLEEHLLCHRFFKNALEIASEILQTPVTEYDLYSHTSIWLLRTDFVLDYPKPVMPNMIFIGGINCHQGKPLPMEFEAYINASGEHGIVVFSLGSMVSEIPEKKAMAIADALGKIPQTVLWRYTGTRPSNLANNTILVKWLPQNDLLGHPMTRAFITHAGSHGVYESICNGVPMVMMPLFGDQMDNAKRMETKGAGVTLNVLEMTSEDLENALKAVINDKSYKENIMRLSSLHKDRPVEPLDLAVFWVEFVMRHKGAPHLRPAAHDLTWYQYHSLDVIGFLLAVVLTVAFITFKCCAYGYRKCLGKKGRVKKAHKSKTH"
    },
    {
        "id": "UGT2B4",
        "sequence": "MSMKWTSALLLIQLSCYFSSGSCGKVLVWPTEFSHWMNIKTILDELVQRGHEVTVLASSASISFDPNSPSTLKFEVYPVSLTKTEFEDIIKQLVKRWAELPKDTFWSYFSQVQEIMWTFNDILRKFCKDIVSNKKLMKKLQESRFDVVLADAVFPFGELLAELLKIPFVYSLRFSPGYAIEKHSGGLLFPPSYVPVVMSELSDQMTFIERVKNMIYVLYFEFWFQIFDMKKWDQFYSEVLGRPTTLSETMAKADIWLIRNYWDFQFPHPLLPNVEFVGGLHCKPAKPLPKEMEEFVQSSGENGVVVFSLGSMVSNTSEERANVIASALAKIPQKVLWRFDGNKPDTLGLNTRLYKWIPQNDLLGHPKTRAFITHGGANGIYEAIYHGIPMVGVPLFADQPDNIAHMKAKGAAVSLDFHTMSSTDLLNALKTVINDPLYKENAMKLSRIHHDQPVKPLDRAVFWIEFVMRHKGAKHLRVAAHDLTWFQYHSLDVTGFLLACVATVIFIITKCLFCVWKFVRTGKKGKRD"
    },
    {
        "id": "UGT2B7",
        "sequence": "MSVKWTSVILLIQLSFCFSSGNCGKVLVWAAEYSHWMNIKTILDELIQRGHEVTVLASSASILFDPNNSSALKIEIYPTSLTKTELENFIMQQIKRWSDLPKDTFWLYFSQVQEIMSIFGDITRKFCKDVVSNKKFMKKVQESRFDVIFADAIFPCSELLAELFNIPFVYSLSFSPGYTFEKHSGGFIFPPSYVPVVMSELTDQMTFMERVKNMIYVLYFDFWFEIFDMKKWDQFYSEVLGRPTTLSETMGKADVWLIRNSWNFQFPYPLLPNVDFVGGLHCKPAKPLPKEMEDFVQSSGENGVVVFSLGSMVSNMTEERANVIASALAQIPQKVLWRFDGNKPDTLGLNTRLYKWIPQNDLLGHPKTRAFITHGGANGIYEAIYHGIPMVGIPLFADQPDNIAHMKARGAAVRVDFNTMSSTDLLNALKRVINDPSYKENVMKLSRIQHDQPVKPLDRAVFWIEFVMRHKGAKHLRVAAHDLTWFQYHSLDVIGFLLVCVATVIFIVTKCCLFCFWKFARKAKKGKND"
    }
    # ➕ add up to 10 total
]



In [7]:
#@title Short Protein Sequences Batch
protein_sequences = [
    {
        "id": "Albumin",
        "sequence": "MKWVTFISLLFLFSSAYSRGVFRRDAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL"
    },
    {
        "id": "AGP",
        "sequence": "MALSWVLTVLSLLPLLEAQIPLCANLVPVPITNATLDRITGKWFYIASAFRNEEYNKSVQEIQATFFYFTPNKTEDTIFLREYQTRQDQCIYNTTYLNVQRENGTISRYVGGQEHFAHLLILRDTKTYMLAFDVNDEKNWGLSVYADKPETTKEQLGEFYEALDCLRIPKSDVVYTDWKKDKCEPLEKQHEKERKQEEGES"
    },
    {
        "id": "ABCB1",
        "sequence": "MDLEGDRNGGAKKKNFFKLNNKSEKDKKEKKPTVSVFSMFRYSNWLDKLYMVVGTLAAIIHGAGLPLMMLVFGEMTDIFANAGNLEDLMSNITNRSDINDTGFFMNLEEDMTRYAYYYSGIGAGVLVAAYIQVSFWCLAAGRQIHKIRKQFFHAIMRQEIGWFDVHDVGELNTRLTDDVSKINEGIGDKIGMFFQSMATFFTGFIVGFTRGWKLTLVILAISPVLGLSAAVWAKILSSFTDKELLAYAKAGAVAEEVLAAIRTVIAFGGQKKELERYNKNLEEAKRIGIKKAITANISIGAAFLLIYASYALAFWYGTTLVLSGEYSIGQVLTVFFSVLIGAFSVGQASPSIEAFANARGAAYEIFKIIDNKPSIDSYSKSGHKPDNIKGNLEFRNVHFSYPSRKEVKILKGLNLKVQSGQTVALVGNSGCGKSTTVQLMQRLYDPTEGMVSVDGQDIRTINVRFLREIIGVVSQEPVLFATTIAENIRYGRENVTMDEIEKAVKEANAYDFIMKLPHKFDTLVGERGAQLSGGQKQRIAIARALVRNPKILLLDEATSALDTESEAVVQVALDKARKGRTTIVIAHRLSTVRNADVIAGFDDGVIVEKGNHDELMKEKGIYFKLVTMQTAGNEVELENAADESKSEIDALEMSSNDSRSSLIRKRSTRRSVRGSQAQDRKLSTKEALDESIPPVSFWRIMKLNLTEWPYFVVGVFCAIINGGLQPAFAIIFSKIIGVFTRIDDPETKRQNSNLFSLLFLALGIISFITFFLQGFTFGKAGEILTKRLRYMVFRSMLRQDVSWFDDPKNTTGALTTRLANDAAQVKGAIGSRLAVITQNIANLGTGIIISFIYGWQLTLLLLAIVPIIAIAGVVEMKMLSGQALKDKKELEGAGKIATEAIENFRTVVSLTQEQKFEHMYAQSLQVPYRNSLRKAHIFGITFSFTQAMMYFSYAGCFRFGAYLVAHKLMSFEDVLLVFSAVVFGAMAVGQVSSFAPDYAKAKISAAHIIMIIEKTPLIDSYSTEGLMPNTLEGNVTFGEVVFNYPTRPDIPVLQGLSLEVKKGQTLALVGSSGCGKSTVVQLLERFYDPLAGKVLLDGKEIKRLNVQWLRAHLGIVSQEPILFDCSIAENIAYGDNSRVVSQEEIVRAAKEANIHAFIESLPNKYSTKVGDKGTQLSGGQKQRIAIARALVRQPHILLLDEATSALDTESEKVVQEALDKAREGRTCIVIAHRLSTIQNADLIVVFQNGRVKEHGTHQQLLAQKGIYFSMVSVQAGTKRQ"
    }
]



In [8]:
#@title Loop Function
import time
import requests

def predict_structure(protein_id, sequence, ligand_smiles):
    polymer_id = "A"  # must be <= 4 characters
    template_b64 = template_b64_map.get(protein_id)

    request_body = {
        "polymers": [
            {
                "id": polymer_id,
                "sequence": sequence,
                "molecule_type": "protein",
                "templates": [
                    {
                        "pdb": template_b64,
                        "chain_id": "A"
                    }
                ] if template_b64 else []
            }
        ],
        "ligands": [
            {
                "id": "L1",  # Ligand ID
                "smiles": ligand_smiles,
                "predict_affinity": True  # MUST be True to get scores
            }
        ]
    }

    response = requests.post(
        PUBLIC_URL,
        headers=headers,
        json=request_body
    )

    if response.status_code == 429:
        print("Rate limited. Sleeping 1 second...")
        time.sleep(1)
        return predict_structure(protein_id, sequence, ligand_smiles)

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()

In [9]:
#@title Run Batch 55s/protein
output_dir = Path("boltz2_batch_outputs")
output_dir.mkdir(exist_ok=True)

def run_batch_predictions(protein_batch, ligand_smiles):
    summary_data = []

    for protein in protein_batch:
        protein_id = protein["id"]
        sequence = protein["sequence"]

        print("="*40)
        print("Processing:", protein_id)

        try:
            result = predict_structure(protein_id, sequence, ligand_smiles)

            result_file = f"{protein_id}.json"
            with open(output_dir / result_file, "w") as f:
                json.dump(result, f)

            # ---- EXTRACTION LOGIC ----
            # NIM returns metrics under 'affinity_predictions' keyed by ligand ID ('L1')
            aff_root = result.get("affinity_predictions", result.get("affinities", {}))
            lig_aff = aff_root.get("L1", {})

            # Boltz-2 usually returns lists; grab the first item
            pIC50 = lig_aff.get("affinity_pic50", [None])[0]
            bind_prob = lig_aff.get("affinity_probability_binary", [None])[0]

            summary_data.append({
                "Protein ID": protein_id,
                "pIC50": pIC50,
                "Binding Prob": bind_prob,
                "Result File": result_file
            })

            print(f"Success: pIC50={pIC50}")

        except Exception as e:
            print(f"Error for {protein_id}: {e}")

    return summary_data

summary_data = run_batch_predictions(
    protein_batch=protein_sequences,
    ligand_smiles=ligand
)

Processing: Albumin
Success: pIC50=5.3068125
Processing: AGP
Success: pIC50=6.2019375000000005
Processing: ABCB1
Success: pIC50=4.880562500000001


In [10]:
#@title Block 8: Display scores and structures
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
import py3Dmol

# 1. Display Summary Table
print("\n" + "="*40)
print(" BATCH PREDICTION SUMMARY")
print("="*40)

df = pd.DataFrame(summary_data)
if not df.empty:
    # Add a rounded pIC50 for better reading
    df["pIC50"] = df["pIC50"].apply(lambda x: round(x, 3) if x is not None else "N/A")
    display(HTML(df.to_html(index=False, classes="table table-striped")))

# 2. Visualization Functions
def extract_structure(result):
    if "structures" in result and result["structures"]:
        return result["structures"][0]
    return None

def visualize_top_structure(json_file, protein_id, metrics):
    if not json_file.exists():
        return

    with open(json_file, "r") as f:
        result = json.load(f)

    struct = extract_structure(result)
    if not struct:
        print(f" {protein_id}: No structure found")
        return

    structure_data = struct.get("structure")
    fmt = struct.get("format", "pdb")

    # Display Metrics above the viewer
    print(f"\n" + "-"*40)
    print(f"  {protein_id}")
    print(f"   pIC50: {metrics.get('pIC50', 'N/A')}")
    print(f"   Binding Prob: {metrics.get('Binding Prob', 'N/A')}")
    print("-"*40)
    viewer = py3Dmol.view(width=800, height=400)
    viewer.addModel(structure_data, fmt)
    viewer.setStyle({"cartoon": {"color": "spectrum"}})
    viewer.setStyle({"resn": "LIG1"}, {"stick": {"colorscheme": "magentaCarbon"}})
    viewer.zoomTo()
    viewer.show()

# 3. Run Visualization
for row in summary_data:
    visualize_top_structure(
        json_file=Path("boltz2_batch_outputs") / row["Result File"],
        protein_id=row["Protein ID"],
        metrics=row
    )


 BATCH PREDICTION SUMMARY


Protein ID,pIC50,Binding Prob,Result File
Albumin,5.307,0.208984,Albumin.json
AGP,6.202,0.248047,AGP.json
ABCB1,4.881,0.304688,ABCB1.json



----------------------------------------
  Albumin
   pIC50: 5.3068125
   Binding Prob: 0.208984375
----------------------------------------


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


----------------------------------------
  AGP
   pIC50: 6.2019375000000005
   Binding Prob: 0.248046875
----------------------------------------


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


----------------------------------------
  ABCB1
   pIC50: 4.880562500000001
   Binding Prob: 0.3046875
----------------------------------------


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
#@title Export Results
import pandas as pd
import zipfile
import json
from pathlib import Path
from google.colab import files

# 1. Setup Paths
output_dir = Path("boltz2_batch_outputs")
pdb_export_dir = Path("exported_pdbs")
pdb_export_dir.mkdir(exist_ok=True)
csv_filename = "affinity_summary.csv"
zip_filename = "boltz2_structures_only.zip"

# 2. Export CSV Summary
if 'summary_data' in locals() and summary_data:
    df_export = pd.DataFrame(summary_data)

    # Calculate micromolar concentration (IC50)
    def to_um(pic50):
        try:
            return round(10 ** (6 - pic50), 3) if pic50 is not None else None
        except:
            return None

    df_export["IC50_uM"] = df_export["pIC50"].apply(to_um)
    df_export.to_csv(csv_filename, index=False)
    print(f" Created {csv_filename}")
else:
    print(" No summary data found.")

# 3. Extract PDB strings from JSON and Save as .pdb files
print(" Extracting PDBs from JSON results...")
for json_file in output_dir.glob("*.json"):
    try:
        with open(json_file, "r") as f:
            data = json.load(f)

        # Look for structure data
        struct_entry = None
        if "structures" in data and data["structures"]:
            struct_entry = data["structures"][0]
        elif "predictions" in data and data["predictions"]:
            struct_entry = data["predictions"][0]

        if struct_entry and "structure" in struct_entry:
            pdb_string = struct_entry["structure"]
            protein_id = json_file.stem # Gets filename without .json
            pdb_path = pdb_export_dir / f"{protein_id}.pdb"

            with open(pdb_path, "w") as f_out:
                f_out.write(pdb_string)
    except Exception as e:
        print(f" Could not extract PDB from {json_file.name}: {e}")

# 4. Create ZIP of PDB files only
try:
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        pdb_files = list(pdb_export_dir.glob("*.pdb"))
        for pdb_file in pdb_files:
            zipf.write(pdb_file, arcname=pdb_file.name)
    print(f" Created {zip_filename} containing {len(pdb_files)} PDB files.")
except Exception as e:
    print(f" Failed to create zip: {e}")

# 5. Optional: Automatic Download
files.download(csv_filename)
files.download(zip_filename)

# Optional extract zip
extract_dir = 'extracted_contents' # Directory to extract to
with zipfile.ZipFile(zip_filename, 'r') as archive:
    archive.extractall(path=extract_dir)

 Created affinity_summary.csv
 Extracting PDBs from JSON results...
 Created boltz2_structures_only.zip containing 3 PDB files.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>